#Experimentation Spatial Attention

##Kernel 3 Blur Kernel 3

In [ ]:
# ===============================================
# 0. Imports
# ===============================================
import os, zipfile
from glob import glob
import cv2
import numpy as np
import matplotlib.pyplot as plt
from tqdm import tqdm
import tensorflow as tf
from tensorflow.keras import layers, models
from sklearn.model_selection import KFold
import tensorflow.keras.backend as K
from sklearn.utils import shuffle
import albumentations as A

# ✅ Enable mixed precision
from tensorflow.keras import mixed_precision
mixed_precision.set_global_policy("mixed_float16")

# ===============================================
# 1. Unzip Dataset + Auto Path Fix
# ===============================================
zip_path = "/content/stage1_train.zip"
extract_root = "/content/stage1_train"
os.makedirs(extract_root, exist_ok=True)

if not any(os.path.isdir(os.path.join(extract_root, d)) for d in os.listdir(extract_root)):
    print("📦 Extracting dataset...")
    with zipfile.ZipFile(zip_path, 'r') as zip_ref:
        zip_ref.extractall(extract_root)
    print("✅ Extraction completed.")
else:
    print("✅ Dataset already extracted.")

candidates = [d for d in glob(os.path.join(extract_root, "*")) if os.path.isdir(d)]
extract_path = candidates[0] if len(candidates) == 1 else extract_root
print("✅ Using dataset folder:", extract_path)
print("🔍 First 5 contents:", os.listdir(extract_path)[:5])

# ===============================================
# 2. Load Data Function
# ===============================================
def load_data(data_dir, img_size=(512, 512)):
    images, masks = [], []
    all_ids = [d for d in glob(os.path.join(data_dir, "*")) if os.path.isdir(d)]
    print(f"🔍 Found {len(all_ids)} patient folders")

    for img_id in tqdm(all_ids, desc="Loading patients"):
        img_path_list = glob(os.path.join(img_id, "images", "*.png"))
        mask_paths = glob(os.path.join(img_id, "masks", "*.png"))
        if len(img_path_list) == 0 or len(mask_paths) == 0:
            continue

        image = cv2.imread(img_path_list[0], cv2.IMREAD_COLOR)
        if image is None:
            continue
        image = cv2.resize(image, img_size)
        image = image / 255.0

        mask = np.zeros(img_size, dtype=np.uint8)
        for mp in mask_paths:
            m = cv2.imread(mp, cv2.IMREAD_GRAYSCALE)
            if m is None:
                continue
            m = cv2.resize(m, img_size)
            mask = np.maximum(mask, m)
        mask = (mask > 127).astype(np.float32)
        mask = np.expand_dims(mask, axis=-1)

        images.append(image)
        masks.append(mask)

    return np.array(images, dtype=np.float32), np.array(masks, dtype=np.float32)

# ===============================================
# 3. Load Dataset
# ===============================================
IMG_SIZE = (512, 512)
images, masks = load_data(extract_path, img_size=IMG_SIZE)
print("✅ Images:", images.shape)
print("✅ Masks:", masks.shape)

# ===============================================
# 4. Loss Functions
# ===============================================
def dice_loss(y_true, y_pred, smooth=1e-6):
    y_true_f = K.flatten(y_true)
    y_pred_f = K.flatten(y_pred)
    intersection = K.sum(y_true_f * y_pred_f)
    return 1 - (2. * intersection + smooth) / (K.sum(y_true_f) + K.sum(y_pred_f) + smooth)

def bce_dice_loss(y_true, y_pred):
    bce = tf.keras.losses.binary_crossentropy(y_true, y_pred)
    dice = dice_loss(y_true, y_pred)
    return bce + dice

# ===============================================
# 5. Convolution Blur Attention (CBA)
# ===============================================
class ConvBlurAttention(layers.Layer):
    def __init__(self, filters, kernel_size=3, activation=tf.nn.leaky_relu):
        super().__init__()
        self.conv = layers.Conv2D(filters, kernel_size, padding="same")
        self.blur = layers.DepthwiseConv2D(kernel_size=3, padding="same", use_bias=False,
                                           depthwise_initializer=tf.keras.initializers.Constant(1/9))
        self.attn = layers.Conv2D(1, 1, padding="same", activation="sigmoid")
        self.activation = activation

    def call(self, x):
        # Convolution
        conv_out = self.conv(x)
        # Blur (spatial smoothing)
        blur_out = self.blur(conv_out)
        # Attention map
        attn_map = self.attn(blur_out)
        # Weighted output
        out = conv_out * attn_map + blur_out * (1 - attn_map)
        return self.activation(out)

# ===============================================
# 6. Transformer Encoder Components
# ===============================================
class PatchEmbedding(layers.Layer):
    def __init__(self, embed_dim=192):
        super().__init__()
        self.proj = layers.Conv2D(embed_dim, kernel_size=1, strides=1, padding="valid")
        self.flatten = layers.Reshape((-1, embed_dim))

    def call(self, x):
        x = self.proj(x)
        x = self.flatten(x)
        return x

class PositionalEncoding(layers.Layer):
    def __init__(self, num_patches, embed_dim):
        super().__init__()
        self.pos_embed = self.add_weight(
            name="pos_embed",
            shape=(1, num_patches, embed_dim),
            initializer="random_normal"
        )

    def call(self, x):
        return x + self.pos_embed

class TransformerBlock(layers.Layer):
    def __init__(self, embed_dim=192, num_heads=12, mlp_dim=768, dropout=0.1):
        super().__init__()
        self.norm1 = layers.LayerNormalization(epsilon=1e-6)
        self.attn = layers.MultiHeadAttention(num_heads=num_heads, key_dim=embed_dim, dropout=dropout)
        self.drop1 = layers.Dropout(dropout)
        self.norm2 = layers.LayerNormalization(epsilon=1e-6)
        self.mlp = models.Sequential([
            layers.Dense(mlp_dim, activation=tf.nn.gelu),
            layers.Dropout(dropout),
            layers.Dense(embed_dim),
            layers.Dropout(dropout),
        ])

    def call(self, x):
        attn_output = self.attn(self.norm1(x), self.norm1(x))
        x = x + self.drop1(attn_output)
        mlp_output = self.mlp(self.norm2(x))
        return x + mlp_output

def build_transformer_encoder(x, num_layers=4, embed_dim=192, num_heads=12, mlp_dim=768):
    for _ in range(num_layers):
        x = TransformerBlock(embed_dim, num_heads, mlp_dim)(x)
    return x

# ===============================================
# 7. TransUNet with Convolution Blur Attention
# ===============================================
def build_transunet(input_shape=(512, 512, 3), num_classes=1):
    inputs = layers.Input(shape=input_shape)
    base_model = tf.keras.applications.ResNet50(include_top=False, weights="imagenet", input_tensor=inputs)
    skip1 = base_model.get_layer("conv1_relu").output
    skip2 = base_model.get_layer("conv2_block3_out").output
    skip3 = base_model.get_layer("conv3_block4_out").output
    cnn_feature = base_model.get_layer("conv4_block6_out").output

    patches = PatchEmbedding(embed_dim=192)(cnn_feature)
    num_patches = patches.shape[1]
    patches = PositionalEncoding(num_patches, 192)(patches)
    transformer_out = build_transformer_encoder(patches)

    h, w = cnn_feature.shape[1], cnn_feature.shape[2]
    x = layers.Reshape((h, w, 192))(transformer_out)

    # Decoder with Convolution Blur Attention
    x = layers.Conv2DTranspose(512, 3, strides=2, padding="same")(x)
    x = layers.Concatenate()([x, skip3])
    x = ConvBlurAttention(512)(x)

    x = layers.Conv2DTranspose(256, 3, strides=2, padding="same")(x)
    x = layers.Concatenate()([x, skip2])
    x = ConvBlurAttention(256)(x)

    x = layers.Conv2DTranspose(128, 3, strides=2, padding="same")(x)
    x = layers.Concatenate()([x, skip1])
    x = ConvBlurAttention(128)(x)

    x = layers.Conv2DTranspose(64, 3, strides=2, padding="same")(x)
    x = ConvBlurAttention(64)(x)

    outputs = layers.Conv2D(num_classes, 1, activation="sigmoid", dtype="float32")(x)
    return models.Model(inputs, outputs, name="TransUNet_CBA")

# ===============================================
# 8. Metrics
# ===============================================
def calculate_metrics(y_true, y_pred, threshold=0.5, smooth=1e-6):
    y_pred_bin = (y_pred > threshold).astype(np.float32)
    pixel_acc = np.mean(y_true == y_pred_bin)
    intersection = np.sum(y_true * y_pred_bin)
    dice = (2 * intersection + smooth) / (np.sum(y_true) + np.sum(y_pred_bin) + smooth)
    union = np.sum(y_true) + np.sum(y_pred_bin) - intersection
    iou = (intersection + smooth) / (union + smooth)
    return pixel_acc, dice, iou

# ===============================================
# 9. K-Fold Training
# ===============================================
images, masks = shuffle(images, masks, random_state=42)
num_folds = 5
kf = KFold(n_splits=num_folds, shuffle=True, random_state=42)
fold_metrics = []

augmentations = [
    A.HorizontalFlip(p=1.0),
    A.VerticalFlip(p=1.0),
    A.GaussianBlur(blur_limit=(3,3), sigma_limit=2, p=1.0),
    A.GaussianBlur(blur_limit=(5,5), sigma_limit=4, p=1.0),
    A.Rotate(limit=270, interpolation=cv2.INTER_LINEAR, border_mode=cv2.BORDER_REFLECT_101, p=1.0),
]

for fold, (train_idx, val_idx) in enumerate(kf.split(images)):
    print(f"\n📂 Fold {fold+1}/{num_folds}")
    X_train, X_val = images[train_idx], images[val_idx]
    y_train, y_val = masks[train_idx], masks[val_idx]

    aug_X, aug_y = [], []
    print("🔄 Applying augmentations...")
    for img, mask in tqdm(zip(X_train, y_train), total=len(X_train)):
        aug_X.append(img)
        aug_y.append(mask)
        for aug in augmentations:
            augmented = aug(image=(img*255).astype(np.uint8), mask=(mask*255).astype(np.uint8))
            img_aug = augmented["image"].astype(np.float32)/255.0
            mask_aug = (augmented["mask"]>127).astype(np.float32)
            img_aug = np.reshape(img_aug, (512,512,3))
            mask_aug = np.reshape(mask_aug, (512,512,1))
            aug_X.append(img_aug)
            aug_y.append(mask_aug)

    X_train = np.array(aug_X, dtype=np.float32)
    y_train = np.array(aug_y, dtype=np.float32)
    print(f"✅ Augmented training set: {X_train.shape}, {y_train.shape}")

    model = build_transunet(input_shape=(512,512,3))
    model.compile(optimizer=tf.keras.optimizers.Adam(1e-4), loss=bce_dice_loss, metrics=["accuracy"])

    callbacks = [
        tf.keras.callbacks.ModelCheckpoint(
            f"transunet_cba_fold{fold+1}.weights.h5",
            save_best_only=True,
            monitor="val_loss"
        )
    ]

    model.fit(
        X_train, y_train,
        validation_data=(X_val, y_val),
        epochs=50,
        batch_size=16,
        callbacks=callbacks,
        verbose=1
    )

    preds_val = model.predict(X_val)
    pixel_acc, dice_score, iou_score = calculate_metrics(y_val, preds_val)
    print(f"✅ Fold {fold+1} | Pixel Acc: {pixel_acc:.4f} | Dice: {dice_score:.4f} | IoU: {iou_score:.4f}")
    fold_metrics.append((pixel_acc, dice_score, iou_score))

# ===============================================
# 10. Final Results
# ===============================================
fold_metrics = np.array(fold_metrics)
mean_metrics = np.mean(fold_metrics, axis=0)
std_metrics = np.std(fold_metrics, axis=0)
print("\n📊 Final Cross-Validation Results:")
print(f"Pixel Accuracy: {mean_metrics[0]:.4f} ± {std_metrics[0]:.4f}")
print(f"Dice Score:     {mean_metrics[1]:.4f} ± {std_metrics[1]:.4f}")
print(f"IoU Score:      {mean_metrics[2]:.4f} ± {std_metrics[2]:.4f}")

📦 Extracting dataset...
✅ Extraction completed.
✅ Using dataset folder: /content/stage1_train
🔍 First 5 contents: ['9ebcfaf2322932d464f15b5662cae4d669b2d785b8299556d73fffcae8365d32', 'f7eaaf420b5204c4a42577428b7cd897a53ef07b759ccbba3ed30a3548ca5605', '8e8a7a14749d0b2e48de3d10e2e80063f17b165ad921c8afc0623f08500f3259', '813f41ef376c3cbcc9d6e2ce6a51c2ee068226d1c1b13404eb238dcfdd447c97', '708eb41a3fc8f2b6cd1f529cdf38dc4ad5d5f00ad30bdcba92884f37ff78d614']
🔍 Found 670 patient folders


Loading patients: 100%|██████████| 670/670 [00:29<00:00, 22.80it/s]


✅ Images: (670, 512, 512, 3)
✅ Masks: (670, 512, 512, 1)

📂 Fold 1/5
🔄 Applying augmentations...


100%|██████████| 536/536 [00:12<00:00, 44.44it/s]


✅ Augmented training set: (3216, 512, 512, 3), (3216, 512, 512, 1)
94765736/94765736 ━━━━━━━━━━━━━━━━━━━━ 0s 0us/step
Epoch 1/50
201/201 ━━━━━━━━━━━━━━━━━━━━ 0s 168ms/step - accuracy: 0.9250 - loss: 0.5090

...

201/201 ━━━━━━━━━━━━━━━━━━━━ 36s 178ms/step - accuracy: 0.9797 - loss: 0.1428 - val_accuracy: 0.8952 - val_loss: 1.2768
Epoch 5/50
201/201 ━━━━━━━━━━━━━━━━━━━━ 0s 168ms/step - accuracy: 0.9801 - loss: 0.1404

201/201 ━━━━━━━━━━━━━━━━━━━━ 36s 177ms/step - accuracy: 0.9801 - loss: 0.1404 - val_accuracy: 0.9480 - val_loss: 0.4714
Epoch 6/50
201/201 ━━━━━━━━━━━━━━━━━━━━ 0s 169ms/step - accuracy: 0.9816 - loss: 0.1287

201/201 ━━━━━━━━━━━━━━━━━━━━ 36s 178ms/step - accuracy: 0.9816 - loss: 0.1288 - val_accuracy: 0.9736 - val_loss: 0.2131
Epoch 7/50
201/201 ━━━━━━━━━━━━━━━━━━━━ 0s 167ms/step - accuracy: 0.9817 - loss: 0.1276

201/201 ━━━━━━━━━━━━━━━━━━━━ 35s 176ms/step - accuracy: 0.9817 - loss: 0.1276 - val_accuracy: 0.9803 - val_loss: 0.1462
Epoch 8/50
201/201 ━━━━━━━━━━━━━━━━━━━━ 34s 171ms/step - accuracy: 0.9832 - loss: 0.1190 - val_accuracy: 0.9792 - val_loss: 0.1505
Epoch 9/50
201/201 ━━━━━━━━━━━━━━━━━━━━ 34s 171ms/step - accuracy: 0.9833 - loss: 0.1153 - val_accuracy: 0.9799 - val_loss: 0.1525
Epoch 10/50
201/201 ━━━━━━━━━━━━━━━━━━━━ 34s 171ms/step - accuracy: 0.9846 - loss: 0.1071 - val_accuracy: 0.9802 - val_loss: 0.1471
Epoch 11/50
201/201 ━━━━━━━━━━━━━━━━━━━━ 0s 168ms/step - accuracy: 0.9854 - loss: 0.1013

201/201 ━━━━━━━━━━━━━━━━━━━━ 36s 177ms/step - accuracy: 0.9854 - loss: 0.1013 - val_accuracy: 0.9810 - val_loss: 0.1420
Epoch 12/50
201/201 ━━━━━━━━━━━━━━━━━━━━ 34s 170ms/step - accuracy: 0.9863 - loss: 0.0956 - val_accuracy: 0.9798 - val_loss: 0.1566
Epoch 13/50
201/201 ━━━━━━━━━━━━━━━━━━━━ 34s 171ms/step - accuracy: 0.9864 - loss: 0.0970 - val_accuracy: 0.9808 - val_loss: 0.1489
Epoch 14/50
201/201 ━━━━━━━━━━━━━━━━━━━━ 34s 171ms/step - accuracy: 0.9871 - loss: 0.0893 - val_accuracy: 0.9797 - val_loss: 0.1524
Epoch 15/50
201/201 ━━━━━━━━━━━━━━━━━━━━ 34s 171ms/step - accuracy: 0.9877 - loss: 0.0865 - val_accuracy: 0.9806 - val_loss: 0.1474
Epoch 16/50
201/201 ━━━━━━━━━━━━━━━━━━━━ 34s 171ms/step - accuracy: 0.9883 - loss: 0.0811 - val_accuracy: 0.9801 - val_loss: 0.1570
Epoch 17/50
201/201 ━━━━━━━━━━━━━━━━━━━━ 34s 171ms/step - accuracy: 0.9890 - loss: 0.0765 - val_accuracy: 0.9806 - val_loss: 0.1502
Epoch 18/50
201/201 ━━━━━━━━━━━━━━━━━━━━ 34s 171ms/step - accuracy: 0.9895 - loss: 0.074

##Kernel 5 Blur Kernel 3

In [ ]:
# ===============================================
# 0. Imports
# ===============================================
import os, zipfile
from glob import glob
import cv2
import numpy as np
import matplotlib.pyplot as plt
from tqdm import tqdm
import tensorflow as tf
from tensorflow.keras import layers, models
from sklearn.model_selection import KFold
import tensorflow.keras.backend as K
from sklearn.utils import shuffle
import albumentations as A

# ✅ Enable mixed precision
from tensorflow.keras import mixed_precision
mixed_precision.set_global_policy("mixed_float16")

# ===============================================
# 1. Unzip Dataset + Auto Path Fix
# ===============================================
zip_path = "/content/stage1_train.zip"
extract_root = "/content/stage1_train"
os.makedirs(extract_root, exist_ok=True)

if not any(os.path.isdir(os.path.join(extract_root, d)) for d in os.listdir(extract_root)):
    print("📦 Extracting dataset...")
    with zipfile.ZipFile(zip_path, 'r') as zip_ref:
        zip_ref.extractall(extract_root)
    print("✅ Extraction completed.")
else:
    print("✅ Dataset already extracted.")

candidates = [d for d in glob(os.path.join(extract_root, "*")) if os.path.isdir(d)]
extract_path = candidates[0] if len(candidates) == 1 else extract_root
print("✅ Using dataset folder:", extract_path)
print("🔍 First 5 contents:", os.listdir(extract_path)[:5])

# ===============================================
# 2. Load Data Function
# ===============================================
def load_data(data_dir, img_size=(512, 512)):
    images, masks = [], []
    all_ids = [d for d in glob(os.path.join(data_dir, "*")) if os.path.isdir(d)]
    print(f"🔍 Found {len(all_ids)} patient folders")

    for img_id in tqdm(all_ids, desc="Loading patients"):
        img_path_list = glob(os.path.join(img_id, "images", "*.png"))
        mask_paths = glob(os.path.join(img_id, "masks", "*.png"))
        if len(img_path_list) == 0 or len(mask_paths) == 0:
            continue

        image = cv2.imread(img_path_list[0], cv2.IMREAD_COLOR)
        if image is None:
            continue
        image = cv2.resize(image, img_size)
        image = image / 255.0

        mask = np.zeros(img_size, dtype=np.uint8)
        for mp in mask_paths:
            m = cv2.imread(mp, cv2.IMREAD_GRAYSCALE)
            if m is None:
                continue
            m = cv2.resize(m, img_size)
            mask = np.maximum(mask, m)
        mask = (mask > 127).astype(np.float32)
        mask = np.expand_dims(mask, axis=-1)

        images.append(image)
        masks.append(mask)

    return np.array(images, dtype=np.float32), np.array(masks, dtype=np.float32)

# ===============================================
# 3. Load Dataset
# ===============================================
IMG_SIZE = (512, 512)
images, masks = load_data(extract_path, img_size=IMG_SIZE)
print("✅ Images:", images.shape)
print("✅ Masks:", masks.shape)

# ===============================================
# 4. Loss Functions
# ===============================================
def dice_loss(y_true, y_pred, smooth=1e-6):
    y_true_f = K.flatten(y_true)
    y_pred_f = K.flatten(y_pred)
    intersection = K.sum(y_true_f * y_pred_f)
    return 1 - (2. * intersection + smooth) / (K.sum(y_true_f) + K.sum(y_pred_f) + smooth)

def bce_dice_loss(y_true, y_pred):
    bce = tf.keras.losses.binary_crossentropy(y_true, y_pred)
    dice = dice_loss(y_true, y_pred)
    return bce + dice

# ===============================================
# 5. Convolution Blur Attention (CBA)
# ===============================================
class ConvBlurAttention(layers.Layer):
    def __init__(self, filters, kernel_size=5, activation=tf.nn.leaky_relu):
        super().__init__()
        self.conv = layers.Conv2D(filters, kernel_size, padding="same")
        self.blur = layers.DepthwiseConv2D(kernel_size=3, padding="same", use_bias=False,
                                           depthwise_initializer=tf.keras.initializers.Constant(1/9))
        self.attn = layers.Conv2D(1, 1, padding="same", activation="sigmoid")
        self.activation = activation

    def call(self, x):
        # Convolution
        conv_out = self.conv(x)
        # Blur (spatial smoothing)
        blur_out = self.blur(conv_out)
        # Attention map
        attn_map = self.attn(blur_out)
        # Weighted output
        out = conv_out * attn_map + blur_out * (1 - attn_map)
        return self.activation(out)

# ===============================================
# 6. Transformer Encoder Components
# ===============================================
class PatchEmbedding(layers.Layer):
    def __init__(self, embed_dim=192):
        super().__init__()
        self.proj = layers.Conv2D(embed_dim, kernel_size=1, strides=1, padding="valid")
        self.flatten = layers.Reshape((-1, embed_dim))

    def call(self, x):
        x = self.proj(x)
        x = self.flatten(x)
        return x

class PositionalEncoding(layers.Layer):
    def __init__(self, num_patches, embed_dim):
        super().__init__()
        self.pos_embed = self.add_weight(
            name="pos_embed",
            shape=(1, num_patches, embed_dim),
            initializer="random_normal"
        )

    def call(self, x):
        return x + self.pos_embed

class TransformerBlock(layers.Layer):
    def __init__(self, embed_dim=192, num_heads=12, mlp_dim=768, dropout=0.1):
        super().__init__()
        self.norm1 = layers.LayerNormalization(epsilon=1e-6)
        self.attn = layers.MultiHeadAttention(num_heads=num_heads, key_dim=embed_dim, dropout=dropout)
        self.drop1 = layers.Dropout(dropout)
        self.norm2 = layers.LayerNormalization(epsilon=1e-6)
        self.mlp = models.Sequential([
            layers.Dense(mlp_dim, activation=tf.nn.gelu),
            layers.Dropout(dropout),
            layers.Dense(embed_dim),
            layers.Dropout(dropout),
        ])

    def call(self, x):
        attn_output = self.attn(self.norm1(x), self.norm1(x))
        x = x + self.drop1(attn_output)
        mlp_output = self.mlp(self.norm2(x))
        return x + mlp_output

def build_transformer_encoder(x, num_layers=4, embed_dim=192, num_heads=12, mlp_dim=768):
    for _ in range(num_layers):
        x = TransformerBlock(embed_dim, num_heads, mlp_dim)(x)
    return x

# ===============================================
# 7. TransUNet with Convolution Blur Attention
# ===============================================
def build_transunet(input_shape=(512, 512, 3), num_classes=1):
    inputs = layers.Input(shape=input_shape)
    base_model = tf.keras.applications.ResNet50(include_top=False, weights="imagenet", input_tensor=inputs)
    skip1 = base_model.get_layer("conv1_relu").output
    skip2 = base_model.get_layer("conv2_block3_out").output
    skip3 = base_model.get_layer("conv3_block4_out").output
    cnn_feature = base_model.get_layer("conv4_block6_out").output

    patches = PatchEmbedding(embed_dim=192)(cnn_feature)
    num_patches = patches.shape[1]
    patches = PositionalEncoding(num_patches, 192)(patches)
    transformer_out = build_transformer_encoder(patches)

    h, w = cnn_feature.shape[1], cnn_feature.shape[2]
    x = layers.Reshape((h, w, 192))(transformer_out)

    # Decoder with Convolution Blur Attention
    x = layers.Conv2DTranspose(512, 3, strides=2, padding="same")(x)
    x = layers.Concatenate()([x, skip3])
    x = ConvBlurAttention(512)(x)

    x = layers.Conv2DTranspose(256, 3, strides=2, padding="same")(x)
    x = layers.Concatenate()([x, skip2])
    x = ConvBlurAttention(256)(x)

    x = layers.Conv2DTranspose(128, 3, strides=2, padding="same")(x)
    x = layers.Concatenate()([x, skip1])
    x = ConvBlurAttention(128)(x)

    x = layers.Conv2DTranspose(64, 3, strides=2, padding="same")(x)
    x = ConvBlurAttention(64)(x)

    outputs = layers.Conv2D(num_classes, 1, activation="sigmoid", dtype="float32")(x)
    return models.Model(inputs, outputs, name="TransUNet_CBA")

# ===============================================
# 8. Metrics
# ===============================================
def calculate_metrics(y_true, y_pred, threshold=0.5, smooth=1e-6):
    y_pred_bin = (y_pred > threshold).astype(np.float32)
    pixel_acc = np.mean(y_true == y_pred_bin)
    intersection = np.sum(y_true * y_pred_bin)
    dice = (2 * intersection + smooth) / (np.sum(y_true) + np.sum(y_pred_bin) + smooth)
    union = np.sum(y_true) + np.sum(y_pred_bin) - intersection
    iou = (intersection + smooth) / (union + smooth)
    return pixel_acc, dice, iou

# ===============================================
# 9. K-Fold Training
# ===============================================
images, masks = shuffle(images, masks, random_state=42)
num_folds = 5
kf = KFold(n_splits=num_folds, shuffle=True, random_state=42)
fold_metrics = []

augmentations = [
    A.HorizontalFlip(p=1.0),
    A.VerticalFlip(p=1.0),
    A.GaussianBlur(blur_limit=(3,3), sigma_limit=2, p=1.0),
    A.GaussianBlur(blur_limit=(5,5), sigma_limit=4, p=1.0),
    A.Rotate(limit=270, interpolation=cv2.INTER_LINEAR, border_mode=cv2.BORDER_REFLECT_101, p=1.0),
]

for fold, (train_idx, val_idx) in enumerate(kf.split(images)):
    print(f"\n📂 Fold {fold+1}/{num_folds}")
    X_train, X_val = images[train_idx], images[val_idx]
    y_train, y_val = masks[train_idx], masks[val_idx]

    aug_X, aug_y = [], []
    print("🔄 Applying augmentations...")
    for img, mask in tqdm(zip(X_train, y_train), total=len(X_train)):
        aug_X.append(img)
        aug_y.append(mask)
        for aug in augmentations:
            augmented = aug(image=(img*255).astype(np.uint8), mask=(mask*255).astype(np.uint8))
            img_aug = augmented["image"].astype(np.float32)/255.0
            mask_aug = (augmented["mask"]>127).astype(np.float32)
            img_aug = np.reshape(img_aug, (512,512,3))
            mask_aug = np.reshape(mask_aug, (512,512,1))
            aug_X.append(img_aug)
            aug_y.append(mask_aug)

    X_train = np.array(aug_X, dtype=np.float32)
    y_train = np.array(aug_y, dtype=np.float32)
    print(f"✅ Augmented training set: {X_train.shape}, {y_train.shape}")

    model = build_transunet(input_shape=(512,512,3))
    model.compile(optimizer=tf.keras.optimizers.Adam(1e-4), loss=bce_dice_loss, metrics=["accuracy"])

    callbacks = [
        tf.keras.callbacks.ModelCheckpoint(
            f"transunet_cbak5b3_fold{fold+1}.weights.h5",
            save_best_only=True,
            monitor="val_loss"
        )
    ]

    model.fit(
        X_train, y_train,
        validation_data=(X_val, y_val),
        epochs=50,
        batch_size=16,
        callbacks=callbacks,
        verbose=1
    )

    preds_val = model.predict(X_val)
    pixel_acc, dice_score, iou_score = calculate_metrics(y_val, preds_val)
    print(f"✅ Fold {fold+1} | Pixel Acc: {pixel_acc:.4f} | Dice: {dice_score:.4f} | IoU: {iou_score:.4f}")
    fold_metrics.append((pixel_acc, dice_score, iou_score))

# ===============================================
# 10. Final Results
# ===============================================
fold_metrics = np.array(fold_metrics)
mean_metrics = np.mean(fold_metrics, axis=0)
std_metrics = np.std(fold_metrics, axis=0)
print("\n📊 Final Cross-Validation Results:")
print(f"Pixel Accuracy: {mean_metrics[0]:.4f} ± {std_metrics[0]:.4f}")
print(f"Dice Score:     {mean_metrics[1]:.4f} ± {std_metrics[1]:.4f}")
print(f"IoU Score:      {mean_metrics[2]:.4f} ± {std_metrics[2]:.4f}")

📦 Extracting dataset...
✅ Extraction completed.
✅ Using dataset folder: /content/stage1_train
🔍 First 5 contents: ['3ab9cab6212fabd723a2c5a1949c2ded19980398b56e6080978e796f45cbbc90', 'eeb142344e9de3250ab748f93940bf06be70d5078337680998468a134a101698', 'c96109cbebcf206f20035cbde414e43872074eee8d839ba214feed9cd36277a1', '91cc2e0d4d6e2c1ad59a8d63bcbe3e2ea8bc7f8e642e942a0113450181e73379', 'f43169e3d8b4f71e687945b9e72cbfdfe2e40e68842568e6a30c60d64c1378b6']
🔍 Found 670 patient folders


Loading patients: 100%|██████████| 670/670 [00:29<00:00, 22.97it/s]


✅ Images: (670, 512, 512, 3)
✅ Masks: (670, 512, 512, 1)

📂 Fold 1/5
🔄 Applying augmentations...


100%|██████████| 536/536 [00:11<00:00, 44.84it/s]


✅ Augmented training set: (3216, 512, 512, 3), (3216, 512, 512, 1)
94765736/94765736 ━━━━━━━━━━━━━━━━━━━━ 5s 0us/step
Epoch 1/50
201/201 ━━━━━━━━━━━━━━━━━━━━ 0s 209ms/step - accuracy: 0.9174 - loss: 0.5044

...

201/201 ━━━━━━━━━━━━━━━━━━━━ 45s 223ms/step - accuracy: 0.9771 - loss: 0.1661 - val_accuracy: 0.8519 - val_loss: 2.4645
Epoch 3/50
201/201 ━━━━━━━━━━━━━━━━━━━━ 43s 215ms/step - accuracy: 0.9790 - loss: 0.1503 - val_accuracy: 0.8520 - val_loss: 2.7278
Epoch 4/50
201/201 ━━━━━━━━━━━━━━━━━━━━ 0s 211ms/step - accuracy: 0.9803 - loss: 0.1421

201/201 ━━━━━━━━━━━━━━━━━━━━ 45s 222ms/step - accuracy: 0.9803 - loss: 0.1421 - val_accuracy: 0.8655 - val_loss: 2.0320
Epoch 5/50
201/201 ━━━━━━━━━━━━━━━━━━━━ 0s 211ms/step - accuracy: 0.9813 - loss: 0.1341

201/201 ━━━━━━━━━━━━━━━━━━━━ 45s 221ms/step - accuracy: 0.9813 - loss: 0.1341 - val_accuracy: 0.9167 - val_loss: 0.8208
Epoch 6/50
201/201 ━━━━━━━━━━━━━━━━━━━━ 0s 211ms/step - accuracy: 0.9832 - loss: 0.1209

201/201 ━━━━━━━━━━━━━━━━━━━━ 44s 221ms/step - accuracy: 0.9832 - loss: 0.1209 - val_accuracy: 0.9744 - val_loss: 0.1897
Epoch 7/50
201/201 ━━━━━━━━━━━━━━━━━━━━ 0s 211ms/step - accuracy: 0.9836 - loss: 0.1183

201/201 ━━━━━━━━━━━━━━━━━━━━ 45s 222ms/step - accuracy: 0.9836 - loss: 0.1183 - val_accuracy: 0.9776 - val_loss: 0.1541
Epoch 8/50
201/201 ━━━━━━━━━━━━━━━━━━━━ 43s 216ms/step - accuracy: 0.9840 - loss: 0.1132 - val_accuracy: 0.9761 - val_loss: 0.1681
Epoch 9/50
201/201 ━━━━━━━━━━━━━━━━━━━━ 43s 215ms/step - accuracy: 0.9845 - loss: 0.1109 - val_accuracy: 0.9777 - val_loss: 0.1569
Epoch 10/50
201/201 ━━━━━━━━━━━━━━━━━━━━ 43s 215ms/step - accuracy: 0.9859 - loss: 0.0999 - val_accuracy: 0.9771 - val_loss: 0.1639
Epoch 11/50
201/201 ━━━━━━━━━━━━━━━━━━━━ 43s 215ms/step - accuracy: 0.9861 - loss: 0.1005 - val_accuracy: 0.9769 - val_loss: 0.1686
Epoch 12/50
201/201 ━━━━━━━━━━━━━━━━━━━━ 44s 217ms/step - accuracy: 0.9872 - loss: 0.0906 - val_accuracy: 0.9773 - val_loss: 0.1655
Epoch 13/50
201/201 ━━━━━━━━━━━━━━━━━━━━ 43s 215ms/step - accuracy: 0.9878 - loss: 0.0870 - val_accuracy: 0.9773 - val_loss: 0.1629
Epoch 14/50
201/201 ━━━━━━━━━━━━━━━━━━━━ 43s 215ms/step - accuracy: 0.9889 - loss: 0.0800 

##Kernel 7 Blur 3

In [ ]:
# ===============================================
# 0. Imports
# ===============================================
import os, zipfile
from glob import glob
import cv2
import numpy as np
import matplotlib.pyplot as plt
from tqdm import tqdm
import tensorflow as tf
from tensorflow.keras import layers, models
from sklearn.model_selection import KFold
import tensorflow.keras.backend as K
from sklearn.utils import shuffle
import albumentations as A

# ✅ Enable mixed precision
from tensorflow.keras import mixed_precision
mixed_precision.set_global_policy("mixed_float16")

# ===============================================
# 1. Unzip Dataset + Auto Path Fix
# ===============================================
zip_path = "/content/stage1_train.zip"
extract_root = "/content/stage1_train"
os.makedirs(extract_root, exist_ok=True)

if not any(os.path.isdir(os.path.join(extract_root, d)) for d in os.listdir(extract_root)):
    print("📦 Extracting dataset...")
    with zipfile.ZipFile(zip_path, 'r') as zip_ref:
        zip_ref.extractall(extract_root)
    print("✅ Extraction completed.")
else:
    print("✅ Dataset already extracted.")

candidates = [d for d in glob(os.path.join(extract_root, "*")) if os.path.isdir(d)]
extract_path = candidates[0] if len(candidates) == 1 else extract_root
print("✅ Using dataset folder:", extract_path)
print("🔍 First 5 contents:", os.listdir(extract_path)[:5])

# ===============================================
# 2. Load Data Function
# ===============================================
def load_data(data_dir, img_size=(512, 512)):
    images, masks = [], []
    all_ids = [d for d in glob(os.path.join(data_dir, "*")) if os.path.isdir(d)]
    print(f"🔍 Found {len(all_ids)} patient folders")

    for img_id in tqdm(all_ids, desc="Loading patients"):
        img_path_list = glob(os.path.join(img_id, "images", "*.png"))
        mask_paths = glob(os.path.join(img_id, "masks", "*.png"))
        if len(img_path_list) == 0 or len(mask_paths) == 0:
            continue

        image = cv2.imread(img_path_list[0], cv2.IMREAD_COLOR)
        if image is None:
            continue
        image = cv2.resize(image, img_size)
        image = image / 255.0

        mask = np.zeros(img_size, dtype=np.uint8)
        for mp in mask_paths:
            m = cv2.imread(mp, cv2.IMREAD_GRAYSCALE)
            if m is None:
                continue
            m = cv2.resize(m, img_size)
            mask = np.maximum(mask, m)
        mask = (mask > 127).astype(np.float32)
        mask = np.expand_dims(mask, axis=-1)

        images.append(image)
        masks.append(mask)

    return np.array(images, dtype=np.float32), np.array(masks, dtype=np.float32)

# ===============================================
# 3. Load Dataset
# ===============================================
IMG_SIZE = (512, 512)
images, masks = load_data(extract_path, img_size=IMG_SIZE)
print("✅ Images:", images.shape)
print("✅ Masks:", masks.shape)

# ===============================================
# 4. Loss Functions
# ===============================================
def dice_loss(y_true, y_pred, smooth=1e-6):
    y_true_f = K.flatten(y_true)
    y_pred_f = K.flatten(y_pred)
    intersection = K.sum(y_true_f * y_pred_f)
    return 1 - (2. * intersection + smooth) / (K.sum(y_true_f) + K.sum(y_pred_f) + smooth)

def bce_dice_loss(y_true, y_pred):
    bce = tf.keras.losses.binary_crossentropy(y_true, y_pred)
    dice = dice_loss(y_true, y_pred)
    return bce + dice

# ===============================================
# 5. Convolution Blur Attention (CBA)
# ===============================================
class ConvBlurAttention(layers.Layer):
    def __init__(self, filters, kernel_size=7, activation=tf.nn.leaky_relu):
        super().__init__()
        self.conv = layers.Conv2D(filters, kernel_size, padding="same")
        self.blur = layers.DepthwiseConv2D(kernel_size=3, padding="same", use_bias=False,
                                           depthwise_initializer=tf.keras.initializers.Constant(1/9))
        self.attn = layers.Conv2D(1, 1, padding="same", activation="sigmoid")
        self.activation = activation

    def call(self, x):
        # Convolution
        conv_out = self.conv(x)
        # Blur (spatial smoothing)
        blur_out = self.blur(conv_out)
        # Attention map
        attn_map = self.attn(blur_out)
        # Weighted output
        out = conv_out * attn_map + blur_out * (1 - attn_map)
        return self.activation(out)

# ===============================================
# 6. Transformer Encoder Components
# ===============================================
class PatchEmbedding(layers.Layer):
    def __init__(self, embed_dim=192):
        super().__init__()
        self.proj = layers.Conv2D(embed_dim, kernel_size=1, strides=1, padding="valid")
        self.flatten = layers.Reshape((-1, embed_dim))

    def call(self, x):
        x = self.proj(x)
        x = self.flatten(x)
        return x

class PositionalEncoding(layers.Layer):
    def __init__(self, num_patches, embed_dim):
        super().__init__()
        self.pos_embed = self.add_weight(
            name="pos_embed",
            shape=(1, num_patches, embed_dim),
            initializer="random_normal"
        )

    def call(self, x):
        return x + self.pos_embed

class TransformerBlock(layers.Layer):
    def __init__(self, embed_dim=192, num_heads=12, mlp_dim=768, dropout=0.1):
        super().__init__()
        self.norm1 = layers.LayerNormalization(epsilon=1e-6)
        self.attn = layers.MultiHeadAttention(num_heads=num_heads, key_dim=embed_dim, dropout=dropout)
        self.drop1 = layers.Dropout(dropout)
        self.norm2 = layers.LayerNormalization(epsilon=1e-6)
        self.mlp = models.Sequential([
            layers.Dense(mlp_dim, activation=tf.nn.gelu),
            layers.Dropout(dropout),
            layers.Dense(embed_dim),
            layers.Dropout(dropout),
        ])

    def call(self, x):
        attn_output = self.attn(self.norm1(x), self.norm1(x))
        x = x + self.drop1(attn_output)
        mlp_output = self.mlp(self.norm2(x))
        return x + mlp_output

def build_transformer_encoder(x, num_layers=4, embed_dim=192, num_heads=12, mlp_dim=768):
    for _ in range(num_layers):
        x = TransformerBlock(embed_dim, num_heads, mlp_dim)(x)
    return x

# ===============================================
# 7. TransUNet with Convolution Blur Attention
# ===============================================
def build_transunet(input_shape=(512, 512, 3), num_classes=1):
    inputs = layers.Input(shape=input_shape)
    base_model = tf.keras.applications.ResNet50(include_top=False, weights="imagenet", input_tensor=inputs)
    skip1 = base_model.get_layer("conv1_relu").output
    skip2 = base_model.get_layer("conv2_block3_out").output
    skip3 = base_model.get_layer("conv3_block4_out").output
    cnn_feature = base_model.get_layer("conv4_block6_out").output

    patches = PatchEmbedding(embed_dim=192)(cnn_feature)
    num_patches = patches.shape[1]
    patches = PositionalEncoding(num_patches, 192)(patches)
    transformer_out = build_transformer_encoder(patches)

    h, w = cnn_feature.shape[1], cnn_feature.shape[2]
    x = layers.Reshape((h, w, 192))(transformer_out)

    # Decoder with Convolution Blur Attention
    x = layers.Conv2DTranspose(512, 3, strides=2, padding="same")(x)
    x = layers.Concatenate()([x, skip3])
    x = ConvBlurAttention(512)(x)

    x = layers.Conv2DTranspose(256, 3, strides=2, padding="same")(x)
    x = layers.Concatenate()([x, skip2])
    x = ConvBlurAttention(256)(x)

    x = layers.Conv2DTranspose(128, 3, strides=2, padding="same")(x)
    x = layers.Concatenate()([x, skip1])
    x = ConvBlurAttention(128)(x)

    x = layers.Conv2DTranspose(64, 3, strides=2, padding="same")(x)
    x = ConvBlurAttention(64)(x)

    outputs = layers.Conv2D(num_classes, 1, activation="sigmoid", dtype="float32")(x)
    return models.Model(inputs, outputs, name="TransUNet_CBA")

# ===============================================
# 8. Metrics
# ===============================================
def calculate_metrics(y_true, y_pred, threshold=0.5, smooth=1e-6):
    y_pred_bin = (y_pred > threshold).astype(np.float32)
    pixel_acc = np.mean(y_true == y_pred_bin)
    intersection = np.sum(y_true * y_pred_bin)
    dice = (2 * intersection + smooth) / (np.sum(y_true) + np.sum(y_pred_bin) + smooth)
    union = np.sum(y_true) + np.sum(y_pred_bin) - intersection
    iou = (intersection + smooth) / (union + smooth)
    return pixel_acc, dice, iou

# ===============================================
# 9. K-Fold Training
# ===============================================
images, masks = shuffle(images, masks, random_state=42)
num_folds = 5
kf = KFold(n_splits=num_folds, shuffle=True, random_state=42)
fold_metrics = []

augmentations = [
    A.HorizontalFlip(p=1.0),
    A.VerticalFlip(p=1.0),
    A.GaussianBlur(blur_limit=(3,3), sigma_limit=2, p=1.0),
    A.GaussianBlur(blur_limit=(5,5), sigma_limit=4, p=1.0),
    A.Rotate(limit=270, interpolation=cv2.INTER_LINEAR, border_mode=cv2.BORDER_REFLECT_101, p=1.0),
]

for fold, (train_idx, val_idx) in enumerate(kf.split(images)):
    print(f"\n📂 Fold {fold+1}/{num_folds}")
    X_train, X_val = images[train_idx], images[val_idx]
    y_train, y_val = masks[train_idx], masks[val_idx]

    aug_X, aug_y = [], []
    print("🔄 Applying augmentations...")
    for img, mask in tqdm(zip(X_train, y_train), total=len(X_train)):
        aug_X.append(img)
        aug_y.append(mask)
        for aug in augmentations:
            augmented = aug(image=(img*255).astype(np.uint8), mask=(mask*255).astype(np.uint8))
            img_aug = augmented["image"].astype(np.float32)/255.0
            mask_aug = (augmented["mask"]>127).astype(np.float32)
            img_aug = np.reshape(img_aug, (512,512,3))
            mask_aug = np.reshape(mask_aug, (512,512,1))
            aug_X.append(img_aug)
            aug_y.append(mask_aug)

    X_train = np.array(aug_X, dtype=np.float32)
    y_train = np.array(aug_y, dtype=np.float32)
    print(f"✅ Augmented training set: {X_train.shape}, {y_train.shape}")

    model = build_transunet(input_shape=(512,512,3))
    model.compile(optimizer=tf.keras.optimizers.Adam(1e-4), loss=bce_dice_loss, metrics=["accuracy"])

    callbacks = [
        tf.keras.callbacks.ModelCheckpoint(
            f"transunet_cbak7b3_fold{fold+1}.weights.h5",
            save_best_only=True,
            monitor="val_loss"
        )
    ]

    model.fit(
        X_train, y_train,
        validation_data=(X_val, y_val),
        epochs=50,
        batch_size=16,
        callbacks=callbacks,
        verbose=1
    )

    preds_val = model.predict(X_val)
    pixel_acc, dice_score, iou_score = calculate_metrics(y_val, preds_val)
    print(f"✅ Fold {fold+1} | Pixel Acc: {pixel_acc:.4f} | Dice: {dice_score:.4f} | IoU: {iou_score:.4f}")
    fold_metrics.append((pixel_acc, dice_score, iou_score))

# ===============================================
# 10. Final Results
# ===============================================
fold_metrics = np.array(fold_metrics)
mean_metrics = np.mean(fold_metrics, axis=0)
std_metrics = np.std(fold_metrics, axis=0)
print("\n📊 Final Cross-Validation Results:")
print(f"Pixel Accuracy: {mean_metrics[0]:.4f} ± {std_metrics[0]:.4f}")
print(f"Dice Score:     {mean_metrics[1]:.4f} ± {std_metrics[1]:.4f}")
print(f"IoU Score:      {mean_metrics[2]:.4f} ± {std_metrics[2]:.4f}")

📦 Extracting dataset...
✅ Extraction completed.
✅ Using dataset folder: /content/stage1_train
🔍 First 5 contents: ['3ab9cab6212fabd723a2c5a1949c2ded19980398b56e6080978e796f45cbbc90', 'eeb142344e9de3250ab748f93940bf06be70d5078337680998468a134a101698', 'c96109cbebcf206f20035cbde414e43872074eee8d839ba214feed9cd36277a1', '91cc2e0d4d6e2c1ad59a8d63bcbe3e2ea8bc7f8e642e942a0113450181e73379', 'f43169e3d8b4f71e687945b9e72cbfdfe2e40e68842568e6a30c60d64c1378b6']
🔍 Found 670 patient folders


Loading patients: 100%|██████████| 670/670 [00:28<00:00, 23.26it/s]


✅ Images: (670, 512, 512, 3)
✅ Masks: (670, 512, 512, 1)

📂 Fold 1/5
🔄 Applying augmentations...


100%|██████████| 536/536 [00:11<00:00, 45.27it/s]


✅ Augmented training set: (3216, 512, 512, 3), (3216, 512, 512, 1)
94765736/94765736 ━━━━━━━━━━━━━━━━━━━━ 5s 0us/step
Epoch 1/50
201/201 ━━━━━━━━━━━━━━━━━━━━ 0s 276ms/step - accuracy: 0.9100 - loss: 0.4859

...

201/201 ━━━━━━━━━━━━━━━━━━━━ 59s 292ms/step - accuracy: 0.9803 - loss: 0.1413 - val_accuracy: 0.8687 - val_loss: 1.6475
Epoch 5/50
201/201 ━━━━━━━━━━━━━━━━━━━━ 0s 277ms/step - accuracy: 0.9814 - loss: 0.1349

201/201 ━━━━━━━━━━━━━━━━━━━━ 59s 291ms/step - accuracy: 0.9814 - loss: 0.1349 - val_accuracy: 0.9383 - val_loss: 0.5373
Epoch 6/50
201/201 ━━━━━━━━━━━━━━━━━━━━ 0s 277ms/step - accuracy: 0.9825 - loss: 0.1252

201/201 ━━━━━━━━━━━━━━━━━━━━ 58s 291ms/step - accuracy: 0.9825 - loss: 0.1252 - val_accuracy: 0.9725 - val_loss: 0.2044
Epoch 7/50
201/201 ━━━━━━━━━━━━━━━━━━━━ 0s 278ms/step - accuracy: 0.9833 - loss: 0.1192

201/201 ━━━━━━━━━━━━━━━━━━━━ 59s 291ms/step - accuracy: 0.9833 - loss: 0.1192 - val_accuracy: 0.9776 - val_loss: 0.1574
Epoch 8/50
201/201 ━━━━━━━━━━━━━━━━━━━━ 57s 283ms/step - accuracy: 0.9839 - loss: 0.1136 - val_accuracy: 0.9752 - val_loss: 0.1767
Epoch 9/50
201/201 ━━━━━━━━━━━━━━━━━━━━ 0s 277ms/step - accuracy: 0.9850 - loss: 0.1069

201/201 ━━━━━━━━━━━━━━━━━━━━ 58s 291ms/step - accuracy: 0.9850 - loss: 0.1069 - val_accuracy: 0.9780 - val_loss: 0.1543
Epoch 10/50
201/201 ━━━━━━━━━━━━━━━━━━━━ 57s 284ms/step - accuracy: 0.9862 - loss: 0.0975 - val_accuracy: 0.9781 - val_loss: 0.1575
Epoch 11/50
201/201 ━━━━━━━━━━━━━━━━━━━━ 57s 283ms/step - accuracy: 0.9869 - loss: 0.0927 - val_accuracy: 0.9782 - val_loss: 0.1572
Epoch 12/50
201/201 ━━━━━━━━━━━━━━━━━━━━ 57s 283ms/step - accuracy: 0.9877 - loss: 0.0883 - val_accuracy: 0.9783 - val_loss: 0.1621
Epoch 13/50
201/201 ━━━━━━━━━━━━━━━━━━━━ 57s 282ms/step - accuracy: 0.9885 - loss: 0.0835 - val_accuracy: 0.9780 - val_loss: 0.1673
Epoch 14/50
201/201 ━━━━━━━━━━━━━━━━━━━━ 57s 281ms/step - accuracy: 0.9890 - loss: 0.0782 - val_accuracy: 0.9769 - val_loss: 0.1709
Epoch 15/50
201/201 ━━━━━━━━━━━━━━━━━━━━ 57s 282ms/step - accuracy: 0.9889 - loss: 0.0787 - val_accuracy: 0.9759 - val_loss: 0.1820
Epoch 16/50
201/201 ━━━━━━━━━━━━━━━━━━━━ 57s 282ms/step - accuracy: 0.9893 - loss: 0.077

##Kernel 5 Blur 5

In [ ]:
# ===============================================
# 0. Imports
# ===============================================
import os, zipfile
from glob import glob
import cv2
import numpy as np
import matplotlib.pyplot as plt
from tqdm import tqdm
import tensorflow as tf
from tensorflow.keras import layers, models
from sklearn.model_selection import KFold
import tensorflow.keras.backend as K
from sklearn.utils import shuffle
import albumentations as A

# ✅ Enable mixed precision
from tensorflow.keras import mixed_precision
mixed_precision.set_global_policy("mixed_float16")

# ===============================================
# 1. Unzip Dataset + Auto Path Fix
# ===============================================
zip_path = "/content/stage1_train.zip"
extract_root = "/content/stage1_train"
os.makedirs(extract_root, exist_ok=True)

if not any(os.path.isdir(os.path.join(extract_root, d)) for d in os.listdir(extract_root)):
    print("📦 Extracting dataset...")
    with zipfile.ZipFile(zip_path, 'r') as zip_ref:
        zip_ref.extractall(extract_root)
    print("✅ Extraction completed.")
else:
    print("✅ Dataset already extracted.")

candidates = [d for d in glob(os.path.join(extract_root, "*")) if os.path.isdir(d)]
extract_path = candidates[0] if len(candidates) == 1 else extract_root
print("✅ Using dataset folder:", extract_path)
print("🔍 First 5 contents:", os.listdir(extract_path)[:5])

# ===============================================
# 2. Load Data Function
# ===============================================
def load_data(data_dir, img_size=(512, 512)):
    images, masks = [], []
    all_ids = [d for d in glob(os.path.join(data_dir, "*")) if os.path.isdir(d)]
    print(f"🔍 Found {len(all_ids)} patient folders")

    for img_id in tqdm(all_ids, desc="Loading patients"):
        img_path_list = glob(os.path.join(img_id, "images", "*.png"))
        mask_paths = glob(os.path.join(img_id, "masks", "*.png"))
        if len(img_path_list) == 0 or len(mask_paths) == 0:
            continue

        image = cv2.imread(img_path_list[0], cv2.IMREAD_COLOR)
        if image is None:
            continue
        image = cv2.resize(image, img_size)
        image = image / 255.0

        mask = np.zeros(img_size, dtype=np.uint8)
        for mp in mask_paths:
            m = cv2.imread(mp, cv2.IMREAD_GRAYSCALE)
            if m is None:
                continue
            m = cv2.resize(m, img_size)
            mask = np.maximum(mask, m)
        mask = (mask > 127).astype(np.float32)
        mask = np.expand_dims(mask, axis=-1)

        images.append(image)
        masks.append(mask)

    return np.array(images, dtype=np.float32), np.array(masks, dtype=np.float32)

# ===============================================
# 3. Load Dataset
# ===============================================
IMG_SIZE = (512, 512)
images, masks = load_data(extract_path, img_size=IMG_SIZE)
print("✅ Images:", images.shape)
print("✅ Masks:", masks.shape)

# ===============================================
# 4. Loss Functions
# ===============================================
def dice_loss(y_true, y_pred, smooth=1e-6):
    y_true_f = K.flatten(y_true)
    y_pred_f = K.flatten(y_pred)
    intersection = K.sum(y_true_f * y_pred_f)
    return 1 - (2. * intersection + smooth) / (K.sum(y_true_f) + K.sum(y_pred_f) + smooth)

def bce_dice_loss(y_true, y_pred):
    bce = tf.keras.losses.binary_crossentropy(y_true, y_pred)
    dice = dice_loss(y_true, y_pred)
    return bce + dice

# ===============================================
# 5. Convolution Blur Attention (CBA)
# ===============================================
class ConvBlurAttention(layers.Layer):
    def __init__(self, filters, kernel_size=5, activation=tf.nn.leaky_relu):
        super().__init__()
        self.conv = layers.Conv2D(filters, kernel_size, padding="same")
        self.blur = layers.DepthwiseConv2D(kernel_size=5, padding="same", use_bias=False,
                                           depthwise_initializer=tf.keras.initializers.Constant(1/9))
        self.attn = layers.Conv2D(1, 1, padding="same", activation="sigmoid")
        self.activation = activation

    def call(self, x):
        # Convolution
        conv_out = self.conv(x)
        # Blur (spatial smoothing)
        blur_out = self.blur(conv_out)
        # Attention map
        attn_map = self.attn(blur_out)
        # Weighted output
        out = conv_out * attn_map + blur_out * (1 - attn_map)
        return self.activation(out)

# ===============================================
# 6. Transformer Encoder Components
# ===============================================
class PatchEmbedding(layers.Layer):
    def __init__(self, embed_dim=192):
        super().__init__()
        self.proj = layers.Conv2D(embed_dim, kernel_size=1, strides=1, padding="valid")
        self.flatten = layers.Reshape((-1, embed_dim))

    def call(self, x):
        x = self.proj(x)
        x = self.flatten(x)
        return x

class PositionalEncoding(layers.Layer):
    def __init__(self, num_patches, embed_dim):
        super().__init__()
        self.pos_embed = self.add_weight(
            name="pos_embed",
            shape=(1, num_patches, embed_dim),
            initializer="random_normal"
        )

    def call(self, x):
        return x + self.pos_embed

class TransformerBlock(layers.Layer):
    def __init__(self, embed_dim=192, num_heads=12, mlp_dim=768, dropout=0.1):
        super().__init__()
        self.norm1 = layers.LayerNormalization(epsilon=1e-6)
        self.attn = layers.MultiHeadAttention(num_heads=num_heads, key_dim=embed_dim, dropout=dropout)
        self.drop1 = layers.Dropout(dropout)
        self.norm2 = layers.LayerNormalization(epsilon=1e-6)
        self.mlp = models.Sequential([
            layers.Dense(mlp_dim, activation=tf.nn.gelu),
            layers.Dropout(dropout),
            layers.Dense(embed_dim),
            layers.Dropout(dropout),
        ])

    def call(self, x):
        attn_output = self.attn(self.norm1(x), self.norm1(x))
        x = x + self.drop1(attn_output)
        mlp_output = self.mlp(self.norm2(x))
        return x + mlp_output

def build_transformer_encoder(x, num_layers=4, embed_dim=192, num_heads=12, mlp_dim=768):
    for _ in range(num_layers):
        x = TransformerBlock(embed_dim, num_heads, mlp_dim)(x)
    return x

# ===============================================
# 7. TransUNet with Convolution Blur Attention
# ===============================================
def build_transunet(input_shape=(512, 512, 3), num_classes=1):
    inputs = layers.Input(shape=input_shape)
    base_model = tf.keras.applications.ResNet50(include_top=False, weights="imagenet", input_tensor=inputs)
    skip1 = base_model.get_layer("conv1_relu").output
    skip2 = base_model.get_layer("conv2_block3_out").output
    skip3 = base_model.get_layer("conv3_block4_out").output
    cnn_feature = base_model.get_layer("conv4_block6_out").output

    patches = PatchEmbedding(embed_dim=192)(cnn_feature)
    num_patches = patches.shape[1]
    patches = PositionalEncoding(num_patches, 192)(patches)
    transformer_out = build_transformer_encoder(patches)

    h, w = cnn_feature.shape[1], cnn_feature.shape[2]
    x = layers.Reshape((h, w, 192))(transformer_out)

    # Decoder with Convolution Blur Attention
    x = layers.Conv2DTranspose(512, 3, strides=2, padding="same")(x)
    x = layers.Concatenate()([x, skip3])
    x = ConvBlurAttention(512)(x)

    x = layers.Conv2DTranspose(256, 3, strides=2, padding="same")(x)
    x = layers.Concatenate()([x, skip2])
    x = ConvBlurAttention(256)(x)

    x = layers.Conv2DTranspose(128, 3, strides=2, padding="same")(x)
    x = layers.Concatenate()([x, skip1])
    x = ConvBlurAttention(128)(x)

    x = layers.Conv2DTranspose(64, 3, strides=2, padding="same")(x)
    x = ConvBlurAttention(64)(x)

    outputs = layers.Conv2D(num_classes, 1, activation="sigmoid", dtype="float32")(x)
    return models.Model(inputs, outputs, name="TransUNet_CBA")

# ===============================================
# 8. Metrics
# ===============================================
def calculate_metrics(y_true, y_pred, threshold=0.5, smooth=1e-6):
    y_pred_bin = (y_pred > threshold).astype(np.float32)
    pixel_acc = np.mean(y_true == y_pred_bin)
    intersection = np.sum(y_true * y_pred_bin)
    dice = (2 * intersection + smooth) / (np.sum(y_true) + np.sum(y_pred_bin) + smooth)
    union = np.sum(y_true) + np.sum(y_pred_bin) - intersection
    iou = (intersection + smooth) / (union + smooth)
    return pixel_acc, dice, iou

# ===============================================
# 9. K-Fold Training
# ===============================================
images, masks = shuffle(images, masks, random_state=42)
num_folds = 5
kf = KFold(n_splits=num_folds, shuffle=True, random_state=42)
fold_metrics = []

augmentations = [
    A.HorizontalFlip(p=1.0),
    A.VerticalFlip(p=1.0),
    A.GaussianBlur(blur_limit=(3,3), sigma_limit=2, p=1.0),
    A.GaussianBlur(blur_limit=(5,5), sigma_limit=4, p=1.0),
    A.Rotate(limit=270, interpolation=cv2.INTER_LINEAR, border_mode=cv2.BORDER_REFLECT_101, p=1.0),
]

for fold, (train_idx, val_idx) in enumerate(kf.split(images)):
    print(f"\n📂 Fold {fold+1}/{num_folds}")
    X_train, X_val = images[train_idx], images[val_idx]
    y_train, y_val = masks[train_idx], masks[val_idx]

    aug_X, aug_y = [], []
    print("🔄 Applying augmentations...")
    for img, mask in tqdm(zip(X_train, y_train), total=len(X_train)):
        aug_X.append(img)
        aug_y.append(mask)
        for aug in augmentations:
            augmented = aug(image=(img*255).astype(np.uint8), mask=(mask*255).astype(np.uint8))
            img_aug = augmented["image"].astype(np.float32)/255.0
            mask_aug = (augmented["mask"]>127).astype(np.float32)
            img_aug = np.reshape(img_aug, (512,512,3))
            mask_aug = np.reshape(mask_aug, (512,512,1))
            aug_X.append(img_aug)
            aug_y.append(mask_aug)

    X_train = np.array(aug_X, dtype=np.float32)
    y_train = np.array(aug_y, dtype=np.float32)
    print(f"✅ Augmented training set: {X_train.shape}, {y_train.shape}")

    model = build_transunet(input_shape=(512,512,3))
    model.compile(optimizer=tf.keras.optimizers.Adam(1e-4), loss=bce_dice_loss, metrics=["accuracy"])

    callbacks = [
        tf.keras.callbacks.ModelCheckpoint(
            f"transunet_cbak5b5_fold{fold+1}.weights.h5",
            save_best_only=True,
            monitor="val_loss"
        )
    ]

    model.fit(
        X_train, y_train,
        validation_data=(X_val, y_val),
        epochs=50,
        batch_size=16,
        callbacks=callbacks,
        verbose=1
    )

    preds_val = model.predict(X_val)
    pixel_acc, dice_score, iou_score = calculate_metrics(y_val, preds_val)
    print(f"✅ Fold {fold+1} | Pixel Acc: {pixel_acc:.4f} | Dice: {dice_score:.4f} | IoU: {iou_score:.4f}")
    fold_metrics.append((pixel_acc, dice_score, iou_score))

# ===============================================
# 10. Final Results
# ===============================================
fold_metrics = np.array(fold_metrics)
mean_metrics = np.mean(fold_metrics, axis=0)
std_metrics = np.std(fold_metrics, axis=0)
print("\n📊 Final Cross-Validation Results:")
print(f"Pixel Accuracy: {mean_metrics[0]:.4f} ± {std_metrics[0]:.4f}")
print(f"Dice Score:     {mean_metrics[1]:.4f} ± {std_metrics[1]:.4f}")
print(f"IoU Score:      {mean_metrics[2]:.4f} ± {std_metrics[2]:.4f}")

📦 Extracting dataset...
✅ Extraction completed.
✅ Using dataset folder: /content/stage1_train
🔍 First 5 contents: ['3ab9cab6212fabd723a2c5a1949c2ded19980398b56e6080978e796f45cbbc90', 'eeb142344e9de3250ab748f93940bf06be70d5078337680998468a134a101698', 'c96109cbebcf206f20035cbde414e43872074eee8d839ba214feed9cd36277a1', '91cc2e0d4d6e2c1ad59a8d63bcbe3e2ea8bc7f8e642e942a0113450181e73379', 'f43169e3d8b4f71e687945b9e72cbfdfe2e40e68842568e6a30c60d64c1378b6']
🔍 Found 670 patient folders


Loading patients: 100%|██████████| 670/670 [00:29<00:00, 22.94it/s]


✅ Images: (670, 512, 512, 3)
✅ Masks: (670, 512, 512, 1)

📂 Fold 1/5
🔄 Applying augmentations...


100%|██████████| 536/536 [00:12<00:00, 44.36it/s]


✅ Augmented training set: (3216, 512, 512, 3), (3216, 512, 512, 1)
94765736/94765736 ━━━━━━━━━━━━━━━━━━━━ 5s 0us/step
Epoch 1/50
201/201 ━━━━━━━━━━━━━━━━━━━━ 0s 212ms/step - accuracy: 0.9109 - loss: 0.5648

...

201/201 ━━━━━━━━━━━━━━━━━━━━ 153s 290ms/step - accuracy: 0.9038 - loss: 0.5895 - val_accuracy: 0.8519 - val_loss: 2.6334
Epoch 2/50
201/201 ━━━━━━━━━━━━━━━━━━━━ 44s 217ms/step - accuracy: 0.9752 - loss: 0.1761 - val_accuracy: 0.8519 - val_loss: 3.1137
Epoch 3/50
201/201 ━━━━━━━━━━━━━━━━━━━━ 44s 217ms/step - accuracy: 0.9791 - loss: 0.1519 - val_accuracy: 0.8526 - val_loss: 2.6420
Epoch 4/50
201/201 ━━━━━━━━━━━━━━━━━━━━ 0s 213ms/step - accuracy: 0.9797 - loss: 0.1471

201/201 ━━━━━━━━━━━━━━━━━━━━ 45s 224ms/step - accuracy: 0.9797 - loss: 0.1471 - val_accuracy: 0.8875 - val_loss: 1.5079
Epoch 5/50
201/201 ━━━━━━━━━━━━━━━━━━━━ 0s 212ms/step - accuracy: 0.9804 - loss: 0.1386

201/201 ━━━━━━━━━━━━━━━━━━━━ 45s 223ms/step - accuracy: 0.9804 - loss: 0.1386 - val_accuracy: 0.9418 - val_loss: 0.4958
Epoch 6/50
201/201 ━━━━━━━━━━━━━━━━━━━━ 0s 213ms/step - accuracy: 0.9817 - loss: 0.1311

201/201 ━━━━━━━━━━━━━━━━━━━━ 45s 223ms/step - accuracy: 0.9817 - loss: 0.1311 - val_accuracy: 0.9765 - val_loss: 0.1671
Epoch 7/50
201/201 ━━━━━━━━━━━━━━━━━━━━ 0s 213ms/step - accuracy: 0.9822 - loss: 0.1246

201/201 ━━━━━━━━━━━━━━━━━━━━ 45s 224ms/step - accuracy: 0.9822 - loss: 0.1246 - val_accuracy: 0.9782 - val_loss: 0.1501
Epoch 8/50
201/201 ━━━━━━━━━━━━━━━━━━━━ 43s 216ms/step - accuracy: 0.9833 - loss: 0.1166 - val_accuracy: 0.9761 - val_loss: 0.1638
Epoch 9/50
201/201 ━━━━━━━━━━━━━━━━━━━━ 44s 216ms/step - accuracy: 0.9843 - loss: 0.1107 - val_accuracy: 0.9782 - val_loss: 0.1502
Epoch 10/50
201/201 ━━━━━━━━━━━━━━━━━━━━ 43s 216ms/step - accuracy: 0.9855 - loss: 0.1026 - val_accuracy: 0.9766 - val_loss: 0.1715
Epoch 11/50
201/201 ━━━━━━━━━━━━━━━━━━━━ 44s 217ms/step - accuracy: 0.9855 - loss: 0.1036 - val_accuracy: 0.9766 - val_loss: 0.1740
Epoch 12/50
201/201 ━━━━━━━━━━━━━━━━━━━━ 44s 216ms/step - accuracy: 0.9859 - loss: 0.0969 - val_accuracy: 0.9777 - val_loss: 0.1617
Epoch 13/50
201/201 ━━━━━━━━━━━━━━━━━━━━ 44s 216ms/step - accuracy: 0.9874 - loss: 0.0891 - val_accuracy: 0.9781 - val_loss: 0.1587
Epoch 14/50
201/201 ━━━━━━━━━━━━━━━━━━━━ 43s 216ms/step - accuracy: 0.9870 - loss: 0.0919 

##Kernel 5 Blur 7

In [ ]:
# ===============================================
# 0. Imports
# ===============================================
import os, zipfile
from glob import glob
import cv2
import numpy as np
import matplotlib.pyplot as plt
from tqdm import tqdm
import tensorflow as tf
from tensorflow.keras import layers, models
from sklearn.model_selection import KFold
import tensorflow.keras.backend as K
from sklearn.utils import shuffle
import albumentations as A

# ✅ Enable mixed precision
from tensorflow.keras import mixed_precision
mixed_precision.set_global_policy("mixed_float16")

# ===============================================
# 1. Unzip Dataset + Auto Path Fix
# ===============================================
zip_path = "/content/stage1_train.zip"
extract_root = "/content/stage1_train"
os.makedirs(extract_root, exist_ok=True)

if not any(os.path.isdir(os.path.join(extract_root, d)) for d in os.listdir(extract_root)):
    print("📦 Extracting dataset...")
    with zipfile.ZipFile(zip_path, 'r') as zip_ref:
        zip_ref.extractall(extract_root)
    print("✅ Extraction completed.")
else:
    print("✅ Dataset already extracted.")

candidates = [d for d in glob(os.path.join(extract_root, "*")) if os.path.isdir(d)]
extract_path = candidates[0] if len(candidates) == 1 else extract_root
print("✅ Using dataset folder:", extract_path)
print("🔍 First 5 contents:", os.listdir(extract_path)[:5])

# ===============================================
# 2. Load Data Function
# ===============================================
def load_data(data_dir, img_size=(512, 512)):
    images, masks = [], []
    all_ids = [d for d in glob(os.path.join(data_dir, "*")) if os.path.isdir(d)]
    print(f"🔍 Found {len(all_ids)} patient folders")

    for img_id in tqdm(all_ids, desc="Loading patients"):
        img_path_list = glob(os.path.join(img_id, "images", "*.png"))
        mask_paths = glob(os.path.join(img_id, "masks", "*.png"))
        if len(img_path_list) == 0 or len(mask_paths) == 0:
            continue

        image = cv2.imread(img_path_list[0], cv2.IMREAD_COLOR)
        if image is None:
            continue
        image = cv2.resize(image, img_size)
        image = image / 255.0

        mask = np.zeros(img_size, dtype=np.uint8)
        for mp in mask_paths:
            m = cv2.imread(mp, cv2.IMREAD_GRAYSCALE)
            if m is None:
                continue
            m = cv2.resize(m, img_size)
            mask = np.maximum(mask, m)
        mask = (mask > 127).astype(np.float32)
        mask = np.expand_dims(mask, axis=-1)

        images.append(image)
        masks.append(mask)

    return np.array(images, dtype=np.float32), np.array(masks, dtype=np.float32)

# ===============================================
# 3. Load Dataset
# ===============================================
IMG_SIZE = (512, 512)
images, masks = load_data(extract_path, img_size=IMG_SIZE)
print("✅ Images:", images.shape)
print("✅ Masks:", masks.shape)

# ===============================================
# 4. Loss Functions
# ===============================================
def dice_loss(y_true, y_pred, smooth=1e-6):
    y_true_f = K.flatten(y_true)
    y_pred_f = K.flatten(y_pred)
    intersection = K.sum(y_true_f * y_pred_f)
    return 1 - (2. * intersection + smooth) / (K.sum(y_true_f) + K.sum(y_pred_f) + smooth)

def bce_dice_loss(y_true, y_pred):
    bce = tf.keras.losses.binary_crossentropy(y_true, y_pred)
    dice = dice_loss(y_true, y_pred)
    return bce + dice

# ===============================================
# 5. Convolution Blur Attention (CBA)
# ===============================================
class ConvBlurAttention(layers.Layer):
    def __init__(self, filters, kernel_size=5, activation=tf.nn.leaky_relu):
        super().__init__()
        self.conv = layers.Conv2D(filters, kernel_size, padding="same")
        self.blur = layers.DepthwiseConv2D(kernel_size=7, padding="same", use_bias=False,
                                           depthwise_initializer=tf.keras.initializers.Constant(1/9))
        self.attn = layers.Conv2D(1, 1, padding="same", activation="sigmoid")
        self.activation = activation

    def call(self, x):
        # Convolution
        conv_out = self.conv(x)
        # Blur (spatial smoothing)
        blur_out = self.blur(conv_out)
        # Attention map
        attn_map = self.attn(blur_out)
        # Weighted output
        out = conv_out * attn_map + blur_out * (1 - attn_map)
        return self.activation(out)

# ===============================================
# 6. Transformer Encoder Components
# ===============================================
class PatchEmbedding(layers.Layer):
    def __init__(self, embed_dim=192):
        super().__init__()
        self.proj = layers.Conv2D(embed_dim, kernel_size=1, strides=1, padding="valid")
        self.flatten = layers.Reshape((-1, embed_dim))

    def call(self, x):
        x = self.proj(x)
        x = self.flatten(x)
        return x

class PositionalEncoding(layers.Layer):
    def __init__(self, num_patches, embed_dim):
        super().__init__()
        self.pos_embed = self.add_weight(
            name="pos_embed",
            shape=(1, num_patches, embed_dim),
            initializer="random_normal"
        )

    def call(self, x):
        return x + self.pos_embed

class TransformerBlock(layers.Layer):
    def __init__(self, embed_dim=192, num_heads=12, mlp_dim=768, dropout=0.1):
        super().__init__()
        self.norm1 = layers.LayerNormalization(epsilon=1e-6)
        self.attn = layers.MultiHeadAttention(num_heads=num_heads, key_dim=embed_dim, dropout=dropout)
        self.drop1 = layers.Dropout(dropout)
        self.norm2 = layers.LayerNormalization(epsilon=1e-6)
        self.mlp = models.Sequential([
            layers.Dense(mlp_dim, activation=tf.nn.gelu),
            layers.Dropout(dropout),
            layers.Dense(embed_dim),
            layers.Dropout(dropout),
        ])

    def call(self, x):
        attn_output = self.attn(self.norm1(x), self.norm1(x))
        x = x + self.drop1(attn_output)
        mlp_output = self.mlp(self.norm2(x))
        return x + mlp_output

def build_transformer_encoder(x, num_layers=4, embed_dim=192, num_heads=12, mlp_dim=768):
    for _ in range(num_layers):
        x = TransformerBlock(embed_dim, num_heads, mlp_dim)(x)
    return x

# ===============================================
# 7. TransUNet with Convolution Blur Attention
# ===============================================
def build_transunet(input_shape=(512, 512, 3), num_classes=1):
    inputs = layers.Input(shape=input_shape)
    base_model = tf.keras.applications.ResNet50(include_top=False, weights="imagenet", input_tensor=inputs)
    skip1 = base_model.get_layer("conv1_relu").output
    skip2 = base_model.get_layer("conv2_block3_out").output
    skip3 = base_model.get_layer("conv3_block4_out").output
    cnn_feature = base_model.get_layer("conv4_block6_out").output

    patches = PatchEmbedding(embed_dim=192)(cnn_feature)
    num_patches = patches.shape[1]
    patches = PositionalEncoding(num_patches, 192)(patches)
    transformer_out = build_transformer_encoder(patches)

    h, w = cnn_feature.shape[1], cnn_feature.shape[2]
    x = layers.Reshape((h, w, 192))(transformer_out)

    # Decoder with Convolution Blur Attention
    x = layers.Conv2DTranspose(512, 3, strides=2, padding="same")(x)
    x = layers.Concatenate()([x, skip3])
    x = ConvBlurAttention(512)(x)

    x = layers.Conv2DTranspose(256, 3, strides=2, padding="same")(x)
    x = layers.Concatenate()([x, skip2])
    x = ConvBlurAttention(256)(x)

    x = layers.Conv2DTranspose(128, 3, strides=2, padding="same")(x)
    x = layers.Concatenate()([x, skip1])
    x = ConvBlurAttention(128)(x)

    x = layers.Conv2DTranspose(64, 3, strides=2, padding="same")(x)
    x = ConvBlurAttention(64)(x)

    outputs = layers.Conv2D(num_classes, 1, activation="sigmoid", dtype="float32")(x)
    return models.Model(inputs, outputs, name="TransUNet_CBA")

# ===============================================
# 8. Metrics
# ===============================================
def calculate_metrics(y_true, y_pred, threshold=0.5, smooth=1e-6):
    y_pred_bin = (y_pred > threshold).astype(np.float32)
    pixel_acc = np.mean(y_true == y_pred_bin)
    intersection = np.sum(y_true * y_pred_bin)
    dice = (2 * intersection + smooth) / (np.sum(y_true) + np.sum(y_pred_bin) + smooth)
    union = np.sum(y_true) + np.sum(y_pred_bin) - intersection
    iou = (intersection + smooth) / (union + smooth)
    return pixel_acc, dice, iou

# ===============================================
# 9. K-Fold Training
# ===============================================
images, masks = shuffle(images, masks, random_state=42)
num_folds = 5
kf = KFold(n_splits=num_folds, shuffle=True, random_state=42)
fold_metrics = []

augmentations = [
    A.HorizontalFlip(p=1.0),
    A.VerticalFlip(p=1.0),
    A.GaussianBlur(blur_limit=(3,3), sigma_limit=2, p=1.0),
    A.GaussianBlur(blur_limit=(5,5), sigma_limit=4, p=1.0),
    A.Rotate(limit=270, interpolation=cv2.INTER_LINEAR, border_mode=cv2.BORDER_REFLECT_101, p=1.0),
]

for fold, (train_idx, val_idx) in enumerate(kf.split(images)):
    print(f"\n📂 Fold {fold+1}/{num_folds}")
    X_train, X_val = images[train_idx], images[val_idx]
    y_train, y_val = masks[train_idx], masks[val_idx]

    aug_X, aug_y = [], []
    print("🔄 Applying augmentations...")
    for img, mask in tqdm(zip(X_train, y_train), total=len(X_train)):
        aug_X.append(img)
        aug_y.append(mask)
        for aug in augmentations:
            augmented = aug(image=(img*255).astype(np.uint8), mask=(mask*255).astype(np.uint8))
            img_aug = augmented["image"].astype(np.float32)/255.0
            mask_aug = (augmented["mask"]>127).astype(np.float32)
            img_aug = np.reshape(img_aug, (512,512,3))
            mask_aug = np.reshape(mask_aug, (512,512,1))
            aug_X.append(img_aug)
            aug_y.append(mask_aug)

    X_train = np.array(aug_X, dtype=np.float32)
    y_train = np.array(aug_y, dtype=np.float32)
    print(f"✅ Augmented training set: {X_train.shape}, {y_train.shape}")

    model = build_transunet(input_shape=(512,512,3))
    model.compile(optimizer=tf.keras.optimizers.Adam(1e-4), loss=bce_dice_loss, metrics=["accuracy"])

    callbacks = [
        tf.keras.callbacks.ModelCheckpoint(
            f"transunet_cbak5b5_fold{fold+1}.weights.h5",
            save_best_only=True,
            monitor="val_loss"
        )
    ]

    model.fit(
        X_train, y_train,
        validation_data=(X_val, y_val),
        epochs=50,
        batch_size=16,
        callbacks=callbacks,
        verbose=1
    )

    preds_val = model.predict(X_val)
    pixel_acc, dice_score, iou_score = calculate_metrics(y_val, preds_val)
    print(f"✅ Fold {fold+1} | Pixel Acc: {pixel_acc:.4f} | Dice: {dice_score:.4f} | IoU: {iou_score:.4f}")
    fold_metrics.append((pixel_acc, dice_score, iou_score))

# ===============================================
# 10. Final Results
# ===============================================
fold_metrics = np.array(fold_metrics)
mean_metrics = np.mean(fold_metrics, axis=0)
std_metrics = np.std(fold_metrics, axis=0)
print("\n📊 Final Cross-Validation Results:")
print(f"Pixel Accuracy: {mean_metrics[0]:.4f} ± {std_metrics[0]:.4f}")
print(f"Dice Score:     {mean_metrics[1]:.4f} ± {std_metrics[1]:.4f}")
print(f"IoU Score:      {mean_metrics[2]:.4f} ± {std_metrics[2]:.4f}")

📦 Extracting dataset...
✅ Extraction completed.
✅ Using dataset folder: /content/stage1_train
🔍 First 5 contents: ['3ab9cab6212fabd723a2c5a1949c2ded19980398b56e6080978e796f45cbbc90', 'eeb142344e9de3250ab748f93940bf06be70d5078337680998468a134a101698', 'c96109cbebcf206f20035cbde414e43872074eee8d839ba214feed9cd36277a1', '91cc2e0d4d6e2c1ad59a8d63bcbe3e2ea8bc7f8e642e942a0113450181e73379', 'f43169e3d8b4f71e687945b9e72cbfdfe2e40e68842568e6a30c60d64c1378b6']
🔍 Found 670 patient folders


Loading patients: 100%|██████████| 670/670 [00:29<00:00, 22.72it/s]


✅ Images: (670, 512, 512, 3)
✅ Masks: (670, 512, 512, 1)

📂 Fold 1/5
🔄 Applying augmentations...


100%|██████████| 536/536 [00:12<00:00, 44.02it/s]


✅ Augmented training set: (3216, 512, 512, 3), (3216, 512, 512, 1)
94765736/94765736 ━━━━━━━━━━━━━━━━━━━━ 0s 0us/step
Epoch 1/50
201/201 ━━━━━━━━━━━━━━━━━━━━ 0s 218ms/step - accuracy: 0.8310 - loss: 2.4810

...

201/201 ━━━━━━━━━━━━━━━━━━━━ 47s 231ms/step - accuracy: 0.9773 - loss: 0.1613 - val_accuracy: 0.9281 - val_loss: 0.5860
Epoch 6/50
201/201 ━━━━━━━━━━━━━━━━━━━━ 0s 222ms/step - accuracy: 0.9789 - loss: 0.1516

201/201 ━━━━━━━━━━━━━━━━━━━━ 47s 233ms/step - accuracy: 0.9789 - loss: 0.1516 - val_accuracy: 0.9722 - val_loss: 0.2027
Epoch 7/50
201/201 ━━━━━━━━━━━━━━━━━━━━ 0s 221ms/step - accuracy: 0.9797 - loss: 0.1488

201/201 ━━━━━━━━━━━━━━━━━━━━ 47s 232ms/step - accuracy: 0.9797 - loss: 0.1487 - val_accuracy: 0.9759 - val_loss: 0.1621
Epoch 8/50
201/201 ━━━━━━━━━━━━━━━━━━━━ 45s 224ms/step - accuracy: 0.9812 - loss: 0.1354 - val_accuracy: 0.9747 - val_loss: 0.1771
Epoch 9/50
201/201 ━━━━━━━━━━━━━━━━━━━━ 45s 225ms/step - accuracy: 0.9820 - loss: 0.1293 - val_accuracy: 0.9725 - val_loss: 0.1900
Epoch 10/50
201/201 ━━━━━━━━━━━━━━━━━━━━ 45s 224ms/step - accuracy: 0.9826 - loss: 0.1224 - val_accuracy: 0.9759 - val_loss: 0.1660
Epoch 11/50
201/201 ━━━━━━━━━━━━━━━━━━━━ 0s 221ms/step - accuracy: 0.9833 - loss: 0.1198

201/201 ━━━━━━━━━━━━━━━━━━━━ 47s 232ms/step - accuracy: 0.9833 - loss: 0.1198 - val_accuracy: 0.9777 - val_loss: 0.1550
Epoch 12/50
201/201 ━━━━━━━━━━━━━━━━━━━━ 45s 225ms/step - accuracy: 0.9839 - loss: 0.1161 - val_accuracy: 0.9778 - val_loss: 0.1581
Epoch 13/50
201/201 ━━━━━━━━━━━━━━━━━━━━ 45s 225ms/step - accuracy: 0.9846 - loss: 0.1089 - val_accuracy: 0.9774 - val_loss: 0.1588
Epoch 14/50
201/201 ━━━━━━━━━━━━━━━━━━━━ 45s 224ms/step - accuracy: 0.9854 - loss: 0.1041 - val_accuracy: 0.9751 - val_loss: 0.1687
Epoch 15/50
201/201 ━━━━━━━━━━━━━━━━━━━━ 0s 221ms/step - accuracy: 0.9858 - loss: 0.1025

201/201 ━━━━━━━━━━━━━━━━━━━━ 47s 232ms/step - accuracy: 0.9858 - loss: 0.1025 - val_accuracy: 0.9782 - val_loss: 0.1549
Epoch 16/50
201/201 ━━━━━━━━━━━━━━━━━━━━ 45s 225ms/step - accuracy: 0.9862 - loss: 0.0983 - val_accuracy: 0.9784 - val_loss: 0.1556
Epoch 17/50
201/201 ━━━━━━━━━━━━━━━━━━━━ 45s 225ms/step - accuracy: 0.9859 - loss: 0.0994 - val_accuracy: 0.9779 - val_loss: 0.1640
Epoch 18/50
201/201 ━━━━━━━━━━━━━━━━━━━━ 45s 225ms/step - accuracy: 0.9869 - loss: 0.0941 - val_accuracy: 0.9782 - val_loss: 0.1602
Epoch 19/50
201/201 ━━━━━━━━━━━━━━━━━━━━ 45s 225ms/step - accuracy: 0.9874 - loss: 0.0909 - val_accuracy: 0.9773 - val_loss: 0.1662
Epoch 20/50
201/201 ━━━━━━━━━━━━━━━━━━━━ 45s 225ms/step - accuracy: 0.9876 - loss: 0.0884 - val_accuracy: 0.9775 - val_loss: 0.1640
Epoch 21/50
201/201 ━━━━━━━━━━━━━━━━━━━━ 45s 224ms/step - accuracy: 0.9879 - loss: 0.0863 - val_accuracy: 0.9772 - val_loss: 0.1719
Epoch 22/50
201/201 ━━━━━━━━━━━━━━━━━━━━ 45s 225ms/step - accuracy: 0.9881 - loss: 0.083

# Hybrid Attention

## Kernel 3 Blur 3 25 Epochs


In [ ]:
# ===============================================
# 0. Imports
# ===============================================
import os, zipfile
from glob import glob
import cv2
import numpy as np
import matplotlib.pyplot as plt
from tqdm import tqdm
import tensorflow as tf
from tensorflow.keras import layers, models
from sklearn.model_selection import KFold
import tensorflow.keras.backend as K
from sklearn.utils import shuffle
import albumentations as A

# ✅ Enable mixed precision
from tensorflow.keras import mixed_precision
mixed_precision.set_global_policy("mixed_float16")

# ===============================================
# 1. Unzip Dataset + Auto Path Fix
# ===============================================
zip_path = "/content/stage1_train.zip"
extract_root = "/content/stage1_train"
os.makedirs(extract_root, exist_ok=True)

if not any(os.path.isdir(os.path.join(extract_root, d)) for d in os.listdir(extract_root)):
    print("📦 Extracting dataset...")
    with zipfile.ZipFile(zip_path, 'r') as zip_ref:
        zip_ref.extractall(extract_root)
    print("✅ Extraction completed.")
else:
    print("✅ Dataset already extracted.")

candidates = [d for d in glob(os.path.join(extract_root, "*")) if os.path.isdir(d)]
extract_path = candidates[0] if len(candidates) == 1 else extract_root
print("✅ Using dataset folder:", extract_path)
print("🔍 First 5 contents:", os.listdir(extract_path)[:5])

# ===============================================
# 2. Load Data Function
# ===============================================
def load_data(data_dir, img_size=(512, 512)):
    images, masks = [], []
    all_ids = [d for d in glob(os.path.join(data_dir, "*")) if os.path.isdir(d)]
    print(f"🔍 Found {len(all_ids)} patient folders")

    for img_id in tqdm(all_ids, desc="Loading patients"):
        img_path_list = glob(os.path.join(img_id, "images", "*.png"))
        mask_paths = glob(os.path.join(img_id, "masks", "*.png"))
        if len(img_path_list) == 0 or len(mask_paths) == 0:
            continue

        image = cv2.imread(img_path_list[0], cv2.IMREAD_COLOR)
        if image is None:
            continue
        image = cv2.resize(image, img_size)
        image = image / 255.0

        mask = np.zeros(img_size, dtype=np.uint8)
        for mp in mask_paths:
            m = cv2.imread(mp, cv2.IMREAD_GRAYSCALE)
            if m is None:
                continue
            m = cv2.resize(m, img_size)
            mask = np.maximum(mask, m)
        mask = (mask > 127).astype(np.float32)
        mask = np.expand_dims(mask, axis=-1)

        images.append(image)
        masks.append(mask)

    return np.array(images, dtype=np.float32), np.array(masks, dtype=np.float32)

# ===============================================
# 3. Load Dataset
# ===============================================
IMG_SIZE = (512, 512)
images, masks = load_data(extract_path, img_size=IMG_SIZE)
print("✅ Images:", images.shape)
print("✅ Masks:", masks.shape)

# ===============================================
# 4. Loss Functions
# ===============================================
def dice_loss(y_true, y_pred, smooth=1e-6):
    y_true_f = K.flatten(y_true)
    y_pred_f = K.flatten(y_pred)
    intersection = K.sum(y_true_f * y_pred_f)
    return 1 - (2. * intersection + smooth) / (K.sum(y_true_f) + K.sum(y_pred_f) + smooth)

def bce_dice_loss(y_true, y_pred):
    bce = tf.keras.losses.binary_crossentropy(y_true, y_pred)
    dice = dice_loss(y_true, y_pred)
    return bce + dice

# ===============================================
# 5. Convolution Blur + Hybrid Attention (CBA)
# ===============================================
class ConvBlurAttention(layers.Layer):
    def __init__(self, filters, kernel_size=3, reduction=16, activation=tf.nn.leaky_relu):
        super().__init__()
        # --- Convolution & Normalization ---
        self.conv = layers.Conv2D(filters, kernel_size, padding="same", use_bias=False)
        self.bn = layers.BatchNormalization()
        # --- Blur ---
        self.blur = layers.DepthwiseConv2D(
            kernel_size=3,
            padding="same",
            use_bias=False,
            depthwise_initializer=tf.keras.initializers.Constant(1/9)
        )
        # --- Channel Attention (SE) ---
        self.global_pool = layers.GlobalAveragePooling2D()
        self.dense1 = layers.Dense(filters // reduction, activation="relu")
        self.dense2 = layers.Dense(filters, activation="sigmoid")
        # --- Spatial Attention ---
        self.spatial_attn = layers.Conv2D(1, 7, padding="same", activation="sigmoid")
        # --- Activation ---
        self.activation = activation

    def call(self, x):
        # 1️⃣ Convolution + Blur
        conv_out = self.conv(x)
        conv_out = self.bn(conv_out)
        blur_out = self.blur(conv_out)

        # 2️⃣ Channel Attention
        se = self.global_pool(blur_out)
        se = self.dense1(se)
        se = self.dense2(se)
        se = tf.reshape(se, [-1, 1, 1, se.shape[-1]])
        ch_attn = conv_out * se

        # 3️⃣ Spatial Attention
        sp_attn_map = self.spatial_attn(ch_attn)
        sp_attn = ch_attn * sp_attn_map

        # 4️⃣ Combine with blurred path
        out = sp_attn * 0.7 + blur_out * 0.3
        return self.activation(out)

# ===============================================
# 6. Transformer Encoder Components
# ===============================================
class PatchEmbedding(layers.Layer):
    def __init__(self, embed_dim=192):
        super().__init__()
        self.proj = layers.Conv2D(embed_dim, kernel_size=1, strides=1, padding="valid")
        self.flatten = layers.Reshape((-1, embed_dim))

    def call(self, x):
        x = self.proj(x)
        x = self.flatten(x)
        return x

class PositionalEncoding(layers.Layer):
    def __init__(self, num_patches, embed_dim):
        super().__init__()
        self.pos_embed = self.add_weight(
            name="pos_embed",
            shape=(1, num_patches, embed_dim),
            initializer="random_normal"
        )

    def call(self, x):
        return x + self.pos_embed

class TransformerBlock(layers.Layer):
    def __init__(self, embed_dim=192, num_heads=12, mlp_dim=768, dropout=0.1):
        super().__init__()
        self.norm1 = layers.LayerNormalization(epsilon=1e-6)
        self.attn = layers.MultiHeadAttention(num_heads=num_heads, key_dim=embed_dim, dropout=dropout)
        self.drop1 = layers.Dropout(dropout)
        self.norm2 = layers.LayerNormalization(epsilon=1e-6)
        self.mlp = models.Sequential([
            layers.Dense(mlp_dim, activation=tf.nn.gelu),
            layers.Dropout(dropout),
            layers.Dense(embed_dim),
            layers.Dropout(dropout),
        ])

    def call(self, x):
        attn_output = self.attn(self.norm1(x), self.norm1(x))
        x = x + self.drop1(attn_output)
        mlp_output = self.mlp(self.norm2(x))
        return x + mlp_output

def build_transformer_encoder(x, num_layers=4, embed_dim=192, num_heads=12, mlp_dim=768):
    for _ in range(num_layers):
        x = TransformerBlock(embed_dim, num_heads, mlp_dim)(x)
    return x

# ===============================================
# 7. TransUNet with Hybrid CBA
# ===============================================
def build_transunet(input_shape=(512, 512, 3), num_classes=1):
    inputs = layers.Input(shape=input_shape)
    base_model = tf.keras.applications.ResNet50(include_top=False, weights="imagenet", input_tensor=inputs)
    skip1 = base_model.get_layer("conv1_relu").output
    skip2 = base_model.get_layer("conv2_block3_out").output
    skip3 = base_model.get_layer("conv3_block4_out").output
    cnn_feature = base_model.get_layer("conv4_block6_out").output

    patches = PatchEmbedding(embed_dim=192)(cnn_feature)
    num_patches = patches.shape[1]
    patches = PositionalEncoding(num_patches, 192)(patches)
    transformer_out = build_transformer_encoder(patches)

    h, w = cnn_feature.shape[1], cnn_feature.shape[2]
    x = layers.Reshape((h, w, 192))(transformer_out)

    x = layers.Conv2DTranspose(512, 3, strides=2, padding="same")(x)
    x = layers.Concatenate()([x, skip3])
    x = ConvBlurAttention(512)(x)

    x = layers.Conv2DTranspose(256, 3, strides=2, padding="same")(x)
    x = layers.Concatenate()([x, skip2])
    x = ConvBlurAttention(256)(x)

    x = layers.Conv2DTranspose(128, 3, strides=2, padding="same")(x)
    x = layers.Concatenate()([x, skip1])
    x = ConvBlurAttention(128)(x)

    x = layers.Conv2DTranspose(64, 3, strides=2, padding="same")(x)
    x = ConvBlurAttention(64)(x)

    outputs = layers.Conv2D(num_classes, 1, activation="sigmoid", dtype="float32")(x)
    return models.Model(inputs, outputs, name="TransUNet_HybridCBA")

# ===============================================
# 8. Metrics
# ===============================================
def calculate_metrics(y_true, y_pred, threshold=0.5, smooth=1e-6):
    y_pred_bin = (y_pred > threshold).astype(np.float32)
    pixel_acc = np.mean(y_true == y_pred_bin)
    intersection = np.sum(y_true * y_pred_bin)
    dice = (2 * intersection + smooth) / (np.sum(y_true) + np.sum(y_pred_bin) + smooth)
    union = np.sum(y_true) + np.sum(y_pred_bin) - intersection
    iou = (intersection + smooth) / (union + smooth)
    return pixel_acc, dice, iou

# ===============================================
# 9. K-Fold Training
# ===============================================
images, masks = shuffle(images, masks, random_state=42)
num_folds = 5
kf = KFold(n_splits=num_folds, shuffle=True, random_state=42)
fold_metrics = []

augmentations = [
    A.HorizontalFlip(p=1.0),
    A.VerticalFlip(p=1.0),
    A.GaussianBlur(blur_limit=(3,3), sigma_limit=2, p=1.0),
    A.GaussianBlur(blur_limit=(7,7), sigma_limit=4, p=1.0),  # ✅ Added Gaussian Blur σ=4
    A.Rotate(limit=270, interpolation=cv2.INTER_LINEAR, border_mode=cv2.BORDER_REFLECT_101, p=1.0),
]

for fold, (train_idx, val_idx) in enumerate(kf.split(images)):
    print(f"\n📂 Fold {fold+1}/{num_folds}")
    X_train, X_val = images[train_idx], images[val_idx]
    y_train, y_val = masks[train_idx], masks[val_idx]

    aug_X, aug_y = [], []
    print("🔄 Applying augmentations...")
    for img, mask in tqdm(zip(X_train, y_train), total=len(X_train)):
        aug_X.append(img)
        aug_y.append(mask)
        for aug in augmentations:
            augmented = aug(image=(img*255).astype(np.uint8), mask=(mask*255).astype(np.uint8))
            img_aug = augmented["image"].astype(np.float32)/255.0
            mask_aug = (augmented["mask"]>127).astype(np.float32)
            img_aug = np.reshape(img_aug, (512,512,3))
            mask_aug = np.reshape(mask_aug, (512,512,1))
            aug_X.append(img_aug)
            aug_y.append(mask_aug)

    X_train = np.array(aug_X, dtype=np.float32)
    y_train = np.array(aug_y, dtype=np.float32)
    print(f"✅ Augmented training set: {X_train.shape}, {y_train.shape}")

    model = build_transunet(input_shape=(512,512,3))
    model.compile(optimizer=tf.keras.optimizers.Adam(1e-4), loss=bce_dice_loss, metrics=["accuracy"])

    callbacks = [
        tf.keras.callbacks.ModelCheckpoint(
            f"transunet_hybridCBA_fold{fold+1}.weights.h5",
            save_best_only=True,
            monitor="val_loss"
        )
    ]

    model.fit(
        X_train, y_train,
        validation_data=(X_val, y_val),
        epochs=25,
        batch_size=16,
        callbacks=callbacks,
        verbose=1
    )

    preds_val = model.predict(X_val)
    pixel_acc, dice_score, iou_score = calculate_metrics(y_val, preds_val)
    print(f"✅ Fold {fold+1} | Pixel Acc: {pixel_acc:.4f} | Dice: {dice_score:.4f} | IoU: {iou_score:.4f}")
    fold_metrics.append((pixel_acc, dice_score, iou_score))

# ===============================================
# 10. Final Results
# ===============================================
fold_metrics = np.array(fold_metrics)
mean_metrics = np.mean(fold_metrics, axis=0)
std_metrics = np.std(fold_metrics, axis=0)
print("\n📊 Final Cross-Validation Results:")
print(f"Pixel Accuracy: {mean_metrics[0]:.4f} ± {std_metrics[0]:.4f}")
print(f"Dice Score:     {mean_metrics[1]:.4f} ± {std_metrics[1]:.4f}")
print(f"IoU Score:      {mean_metrics[2]:.4f} ± {std_metrics[2]:.4f}")

📦 Extracting dataset...
✅ Extraction completed.
✅ Using dataset folder: /content/stage1_train
🔍 First 5 contents: ['3ab9cab6212fabd723a2c5a1949c2ded19980398b56e6080978e796f45cbbc90', 'eeb142344e9de3250ab748f93940bf06be70d5078337680998468a134a101698', 'c96109cbebcf206f20035cbde414e43872074eee8d839ba214feed9cd36277a1', '91cc2e0d4d6e2c1ad59a8d63bcbe3e2ea8bc7f8e642e942a0113450181e73379', 'f43169e3d8b4f71e687945b9e72cbfdfe2e40e68842568e6a30c60d64c1378b6']
🔍 Found 670 patient folders


Loading patients: 100%|██████████| 670/670 [00:29<00:00, 23.08it/s]


✅ Images: (670, 512, 512, 3)
✅ Masks: (670, 512, 512, 1)

📂 Fold 1/5
🔄 Applying augmentations...


100%|██████████| 536/536 [00:12<00:00, 44.26it/s]


✅ Augmented training set: (3216, 512, 512, 3), (3216, 512, 512, 1)
94765736/94765736 ━━━━━━━━━━━━━━━━━━━━ 0s 0us/step
Epoch 1/25
201/201 ━━━━━━━━━━━━━━━━━━━━ 0s 207ms/step - accuracy: 0.9222 - loss: 0.7709

...

201/201 ━━━━━━━━━━━━━━━━━━━━ 44s 219ms/step - accuracy: 0.9779 - loss: 0.2154 - val_accuracy: 0.8756 - val_loss: 1.2552
Epoch 5/25
201/201 ━━━━━━━━━━━━━━━━━━━━ 0s 208ms/step - accuracy: 0.9800 - loss: 0.1740

201/201 ━━━━━━━━━━━━━━━━━━━━ 44s 218ms/step - accuracy: 0.9800 - loss: 0.1740 - val_accuracy: 0.9204 - val_loss: 0.7497
Epoch 6/25
201/201 ━━━━━━━━━━━━━━━━━━━━ 0s 209ms/step - accuracy: 0.9805 - loss: 0.1608

201/201 ━━━━━━━━━━━━━━━━━━━━ 44s 219ms/step - accuracy: 0.9805 - loss: 0.1608 - val_accuracy: 0.9763 - val_loss: 0.1821
Epoch 7/25
201/201 ━━━━━━━━━━━━━━━━━━━━ 43s 212ms/step - accuracy: 0.9809 - loss: 0.1547 - val_accuracy: 0.9726 - val_loss: 0.2048
Epoch 8/25
201/201 ━━━━━━━━━━━━━━━━━━━━ 0s 209ms/step - accuracy: 0.9820 - loss: 0.1384

201/201 ━━━━━━━━━━━━━━━━━━━━ 44s 219ms/step - accuracy: 0.9820 - loss: 0.1384 - val_accuracy: 0.9780 - val_loss: 0.1604
Epoch 9/25
201/201 ━━━━━━━━━━━━━━━━━━━━ 43s 212ms/step - accuracy: 0.9825 - loss: 0.1318 - val_accuracy: 0.9765 - val_loss: 0.1750
Epoch 10/25
201/201 ━━━━━━━━━━━━━━━━━━━━ 43s 212ms/step - accuracy: 0.9829 - loss: 0.1273 - val_accuracy: 0.9769 - val_loss: 0.1612
Epoch 11/25
201/201 ━━━━━━━━━━━━━━━━━━━━ 43s 212ms/step - accuracy: 0.9846 - loss: 0.1160 - val_accuracy: 0.9770 - val_loss: 0.1617
Epoch 12/25
201/201 ━━━━━━━━━━━━━━━━━━━━ 0s 209ms/step - accuracy: 0.9845 - loss: 0.1159

201/201 ━━━━━━━━━━━━━━━━━━━━ 44s 218ms/step - accuracy: 0.9845 - loss: 0.1159 - val_accuracy: 0.9787 - val_loss: 0.1535
Epoch 13/25
201/201 ━━━━━━━━━━━━━━━━━━━━ 43s 212ms/step - accuracy: 0.9852 - loss: 0.1098 - val_accuracy: 0.9784 - val_loss: 0.1551
Epoch 14/25
201/201 ━━━━━━━━━━━━━━━━━━━━ 43s 213ms/step - accuracy: 0.9857 - loss: 0.1049 - val_accuracy: 0.9777 - val_loss: 0.1619
Epoch 15/25
201/201 ━━━━━━━━━━━━━━━━━━━━ 43s 212ms/step - accuracy: 0.9866 - loss: 0.0972 - val_accuracy: 0.9784 - val_loss: 0.1581
Epoch 16/25
201/201 ━━━━━━━━━━━━━━━━━━━━ 43s 212ms/step - accuracy: 0.9869 - loss: 0.0958 - val_accuracy: 0.9763 - val_loss: 0.1696
Epoch 17/25
201/201 ━━━━━━━━━━━━━━━━━━━━ 43s 212ms/step - accuracy: 0.9873 - loss: 0.0926 - val_accuracy: 0.9777 - val_loss: 0.1631
Epoch 18/25
201/201 ━━━━━━━━━━━━━━━━━━━━ 43s 212ms/step - accuracy: 0.9878 - loss: 0.0885 - val_accuracy: 0.9776 - val_loss: 0.1629
Epoch 19/25
201/201 ━━━━━━━━━━━━━━━━━━━━ 43s 212ms/step - accuracy: 0.9884 - loss: 0.084

##Kernel 3 Blur 3 50 Epochs

In [ ]:
# ===============================================
# 0. Imports
# ===============================================
import os, zipfile
from glob import glob
import cv2
import numpy as np
import matplotlib.pyplot as plt
from tqdm import tqdm
import tensorflow as tf
from tensorflow.keras import layers, models
from sklearn.model_selection import KFold
import tensorflow.keras.backend as K
from sklearn.utils import shuffle
import albumentations as A

# ✅ Enable mixed precision
from tensorflow.keras import mixed_precision
mixed_precision.set_global_policy("mixed_float16")

# ===============================================
# 1. Unzip Dataset + Auto Path Fix
# ===============================================
zip_path = "/content/stage1_train.zip"
extract_root = "/content/stage1_train"
os.makedirs(extract_root, exist_ok=True)

if not any(os.path.isdir(os.path.join(extract_root, d)) for d in os.listdir(extract_root)):
    print("📦 Extracting dataset...")
    with zipfile.ZipFile(zip_path, 'r') as zip_ref:
        zip_ref.extractall(extract_root)
    print("✅ Extraction completed.")
else:
    print("✅ Dataset already extracted.")

candidates = [d for d in glob(os.path.join(extract_root, "*")) if os.path.isdir(d)]
extract_path = candidates[0] if len(candidates) == 1 else extract_root
print("✅ Using dataset folder:", extract_path)
print("🔍 First 5 contents:", os.listdir(extract_path)[:5])

# ===============================================
# 2. Load Data Function
# ===============================================
def load_data(data_dir, img_size=(512, 512)):
    images, masks = [], []
    all_ids = [d for d in glob(os.path.join(data_dir, "*")) if os.path.isdir(d)]
    print(f"🔍 Found {len(all_ids)} patient folders")

    for img_id in tqdm(all_ids, desc="Loading patients"):
        img_path_list = glob(os.path.join(img_id, "images", "*.png"))
        mask_paths = glob(os.path.join(img_id, "masks", "*.png"))
        if len(img_path_list) == 0 or len(mask_paths) == 0:
            continue

        image = cv2.imread(img_path_list[0], cv2.IMREAD_COLOR)
        if image is None:
            continue
        image = cv2.resize(image, img_size)
        image = image / 255.0

        mask = np.zeros(img_size, dtype=np.uint8)
        for mp in mask_paths:
            m = cv2.imread(mp, cv2.IMREAD_GRAYSCALE)
            if m is None:
                continue
            m = cv2.resize(m, img_size)
            mask = np.maximum(mask, m)
        mask = (mask > 127).astype(np.float32)
        mask = np.expand_dims(mask, axis=-1)

        images.append(image)
        masks.append(mask)

    return np.array(images, dtype=np.float32), np.array(masks, dtype=np.float32)

# ===============================================
# 3. Load Dataset
# ===============================================
IMG_SIZE = (512, 512)
images, masks = load_data(extract_path, img_size=IMG_SIZE)
print("✅ Images:", images.shape)
print("✅ Masks:", masks.shape)

# ===============================================
# 4. Loss Functions
# ===============================================
def dice_loss(y_true, y_pred, smooth=1e-6):
    y_true_f = K.flatten(y_true)
    y_pred_f = K.flatten(y_pred)
    intersection = K.sum(y_true_f * y_pred_f)
    return 1 - (2. * intersection + smooth) / (K.sum(y_true_f) + K.sum(y_pred_f) + smooth)

def bce_dice_loss(y_true, y_pred):
    bce = tf.keras.losses.binary_crossentropy(y_true, y_pred)
    dice = dice_loss(y_true, y_pred)
    return bce + dice

# ===============================================
# 5. Convolution Blur + Hybrid Attention (CBA)
# ===============================================
class ConvBlurAttention(layers.Layer):
    def __init__(self, filters, kernel_size=3, reduction=16, activation=tf.nn.leaky_relu):
        super().__init__()
        # --- Convolution & Normalization ---
        self.conv = layers.Conv2D(filters, kernel_size, padding="same", use_bias=False)
        self.bn = layers.BatchNormalization()
        # --- Blur ---
        self.blur = layers.DepthwiseConv2D(
            kernel_size=3,
            padding="same",
            use_bias=False,
            depthwise_initializer=tf.keras.initializers.Constant(1/9)
        )
        # --- Channel Attention (SE) ---
        self.global_pool = layers.GlobalAveragePooling2D()
        self.dense1 = layers.Dense(filters // reduction, activation="relu")
        self.dense2 = layers.Dense(filters, activation="sigmoid")
        # --- Spatial Attention ---
        self.spatial_attn = layers.Conv2D(1, 7, padding="same", activation="sigmoid")
        # --- Activation ---
        self.activation = activation

    def call(self, x):
        # 1️⃣ Convolution + Blur
        conv_out = self.conv(x)
        conv_out = self.bn(conv_out)
        blur_out = self.blur(conv_out)

        # 2️⃣ Channel Attention
        se = self.global_pool(blur_out)
        se = self.dense1(se)
        se = self.dense2(se)
        se = tf.reshape(se, [-1, 1, 1, se.shape[-1]])
        ch_attn = conv_out * se

        # 3️⃣ Spatial Attention
        sp_attn_map = self.spatial_attn(ch_attn)
        sp_attn = ch_attn * sp_attn_map

        # 4️⃣ Combine with blurred path
        out = sp_attn * 0.7 + blur_out * 0.3
        return self.activation(out)

# ===============================================
# 6. Transformer Encoder Components
# ===============================================
class PatchEmbedding(layers.Layer):
    def __init__(self, embed_dim=192):
        super().__init__()
        self.proj = layers.Conv2D(embed_dim, kernel_size=1, strides=1, padding="valid")
        self.flatten = layers.Reshape((-1, embed_dim))

    def call(self, x):
        x = self.proj(x)
        x = self.flatten(x)
        return x

class PositionalEncoding(layers.Layer):
    def __init__(self, num_patches, embed_dim):
        super().__init__()
        self.pos_embed = self.add_weight(
            name="pos_embed",
            shape=(1, num_patches, embed_dim),
            initializer="random_normal"
        )

    def call(self, x):
        return x + self.pos_embed

class TransformerBlock(layers.Layer):
    def __init__(self, embed_dim=192, num_heads=12, mlp_dim=768, dropout=0.1):
        super().__init__()
        self.norm1 = layers.LayerNormalization(epsilon=1e-6)
        self.attn = layers.MultiHeadAttention(num_heads=num_heads, key_dim=embed_dim, dropout=dropout)
        self.drop1 = layers.Dropout(dropout)
        self.norm2 = layers.LayerNormalization(epsilon=1e-6)
        self.mlp = models.Sequential([
            layers.Dense(mlp_dim, activation=tf.nn.gelu),
            layers.Dropout(dropout),
            layers.Dense(embed_dim),
            layers.Dropout(dropout),
        ])

    def call(self, x):
        attn_output = self.attn(self.norm1(x), self.norm1(x))
        x = x + self.drop1(attn_output)
        mlp_output = self.mlp(self.norm2(x))
        return x + mlp_output

def build_transformer_encoder(x, num_layers=4, embed_dim=192, num_heads=12, mlp_dim=768):
    for _ in range(num_layers):
        x = TransformerBlock(embed_dim, num_heads, mlp_dim)(x)
    return x

# ===============================================
# 7. TransUNet with Hybrid CBA
# ===============================================
def build_transunet(input_shape=(512, 512, 3), num_classes=1):
    inputs = layers.Input(shape=input_shape)
    base_model = tf.keras.applications.ResNet50(include_top=False, weights="imagenet", input_tensor=inputs)
    skip1 = base_model.get_layer("conv1_relu").output
    skip2 = base_model.get_layer("conv2_block3_out").output
    skip3 = base_model.get_layer("conv3_block4_out").output
    cnn_feature = base_model.get_layer("conv4_block6_out").output

    patches = PatchEmbedding(embed_dim=192)(cnn_feature)
    num_patches = patches.shape[1]
    patches = PositionalEncoding(num_patches, 192)(patches)
    transformer_out = build_transformer_encoder(patches)

    h, w = cnn_feature.shape[1], cnn_feature.shape[2]
    x = layers.Reshape((h, w, 192))(transformer_out)

    x = layers.Conv2DTranspose(512, 3, strides=2, padding="same")(x)
    x = layers.Concatenate()([x, skip3])
    x = ConvBlurAttention(512)(x)

    x = layers.Conv2DTranspose(256, 3, strides=2, padding="same")(x)
    x = layers.Concatenate()([x, skip2])
    x = ConvBlurAttention(256)(x)

    x = layers.Conv2DTranspose(128, 3, strides=2, padding="same")(x)
    x = layers.Concatenate()([x, skip1])
    x = ConvBlurAttention(128)(x)

    x = layers.Conv2DTranspose(64, 3, strides=2, padding="same")(x)
    x = ConvBlurAttention(64)(x)

    outputs = layers.Conv2D(num_classes, 1, activation="sigmoid", dtype="float32")(x)
    return models.Model(inputs, outputs, name="TransUNet_HybridCBA")

# ===============================================
# 8. Metrics
# ===============================================
def calculate_metrics(y_true, y_pred, threshold=0.5, smooth=1e-6):
    y_pred_bin = (y_pred > threshold).astype(np.float32)
    pixel_acc = np.mean(y_true == y_pred_bin)
    intersection = np.sum(y_true * y_pred_bin)
    dice = (2 * intersection + smooth) / (np.sum(y_true) + np.sum(y_pred_bin) + smooth)
    union = np.sum(y_true) + np.sum(y_pred_bin) - intersection
    iou = (intersection + smooth) / (union + smooth)
    return pixel_acc, dice, iou

# ===============================================
# 9. K-Fold Training
# ===============================================
images, masks = shuffle(images, masks, random_state=42)
num_folds = 5
kf = KFold(n_splits=num_folds, shuffle=True, random_state=42)
fold_metrics = []

augmentations = [
    A.HorizontalFlip(p=1.0),
    A.VerticalFlip(p=1.0),
    A.GaussianBlur(blur_limit=(3,3), sigma_limit=2, p=1.0),
    A.GaussianBlur(blur_limit=(7,7), sigma_limit=4, p=1.0),  # ✅ Added Gaussian Blur σ=4
    A.Rotate(limit=270, interpolation=cv2.INTER_LINEAR, border_mode=cv2.BORDER_REFLECT_101, p=1.0),
]

for fold, (train_idx, val_idx) in enumerate(kf.split(images)):
    print(f"\n📂 Fold {fold+1}/{num_folds}")
    X_train, X_val = images[train_idx], images[val_idx]
    y_train, y_val = masks[train_idx], masks[val_idx]

    aug_X, aug_y = [], []
    print("🔄 Applying augmentations...")
    for img, mask in tqdm(zip(X_train, y_train), total=len(X_train)):
        aug_X.append(img)
        aug_y.append(mask)
        for aug in augmentations:
            augmented = aug(image=(img*255).astype(np.uint8), mask=(mask*255).astype(np.uint8))
            img_aug = augmented["image"].astype(np.float32)/255.0
            mask_aug = (augmented["mask"]>127).astype(np.float32)
            img_aug = np.reshape(img_aug, (512,512,3))
            mask_aug = np.reshape(mask_aug, (512,512,1))
            aug_X.append(img_aug)
            aug_y.append(mask_aug)

    X_train = np.array(aug_X, dtype=np.float32)
    y_train = np.array(aug_y, dtype=np.float32)
    print(f"✅ Augmented training set: {X_train.shape}, {y_train.shape}")

    model = build_transunet(input_shape=(512,512,3))
    model.compile(optimizer=tf.keras.optimizers.Adam(1e-4), loss=bce_dice_loss, metrics=["accuracy"])

    callbacks = [
        tf.keras.callbacks.ModelCheckpoint(
            f"transunet_hybridCBA_50epochs_fold{fold+1}.weights.h5",
            save_best_only=True,
            monitor="val_loss"
        )
    ]

    model.fit(
        X_train, y_train,
        validation_data=(X_val, y_val),
        epochs=50,
        batch_size=16,
        callbacks=callbacks,
        verbose=1
    )

    preds_val = model.predict(X_val)
    pixel_acc, dice_score, iou_score = calculate_metrics(y_val, preds_val)
    print(f"✅ Fold {fold+1} | Pixel Acc: {pixel_acc:.4f} | Dice: {dice_score:.4f} | IoU: {iou_score:.4f}")
    fold_metrics.append((pixel_acc, dice_score, iou_score))

# ===============================================
# 10. Final Results
# ===============================================
fold_metrics = np.array(fold_metrics)
mean_metrics = np.mean(fold_metrics, axis=0)
std_metrics = np.std(fold_metrics, axis=0)
print("\n📊 Final Cross-Validation Results:")
print(f"Pixel Accuracy: {mean_metrics[0]:.4f} ± {std_metrics[0]:.4f}")
print(f"Dice Score:     {mean_metrics[1]:.4f} ± {std_metrics[1]:.4f}")
print(f"IoU Score:      {mean_metrics[2]:.4f} ± {std_metrics[2]:.4f}")

📦 Extracting dataset...
✅ Extraction completed.
✅ Using dataset folder: /content/stage1_train
🔍 First 5 contents: ['3ab9cab6212fabd723a2c5a1949c2ded19980398b56e6080978e796f45cbbc90', 'eeb142344e9de3250ab748f93940bf06be70d5078337680998468a134a101698', 'c96109cbebcf206f20035cbde414e43872074eee8d839ba214feed9cd36277a1', '91cc2e0d4d6e2c1ad59a8d63bcbe3e2ea8bc7f8e642e942a0113450181e73379', 'f43169e3d8b4f71e687945b9e72cbfdfe2e40e68842568e6a30c60d64c1378b6']
🔍 Found 670 patient folders


Loading patients: 100%|██████████| 670/670 [00:28<00:00, 23.12it/s]


✅ Images: (670, 512, 512, 3)
✅ Masks: (670, 512, 512, 1)

📂 Fold 1/5
🔄 Applying augmentations...


100%|██████████| 536/536 [00:12<00:00, 43.34it/s]


✅ Augmented training set: (3216, 512, 512, 3), (3216, 512, 512, 1)
94765736/94765736 ━━━━━━━━━━━━━━━━━━━━ 5s 0us/step
Epoch 1/50
201/201 ━━━━━━━━━━━━━━━━━━━━ 0s 196ms/step - accuracy: 0.8951 - loss: 0.9122

...

201/201 ━━━━━━━━━━━━━━━━━━━━ 42s 208ms/step - accuracy: 0.9803 - loss: 0.1921 - val_accuracy: 0.9320 - val_loss: 0.5619
Epoch 6/50
201/201 ━━━━━━━━━━━━━━━━━━━━ 0s 198ms/step - accuracy: 0.9804 - loss: 0.1719

201/201 ━━━━━━━━━━━━━━━━━━━━ 42s 208ms/step - accuracy: 0.9804 - loss: 0.1719 - val_accuracy: 0.9748 - val_loss: 0.1963
Epoch 7/50
201/201 ━━━━━━━━━━━━━━━━━━━━ 0s 199ms/step - accuracy: 0.9811 - loss: 0.1560

201/201 ━━━━━━━━━━━━━━━━━━━━ 42s 209ms/step - accuracy: 0.9811 - loss: 0.1560 - val_accuracy: 0.9778 - val_loss: 0.1696
Epoch 8/50
201/201 ━━━━━━━━━━━━━━━━━━━━ 0s 198ms/step - accuracy: 0.9830 - loss: 0.1334

201/201 ━━━━━━━━━━━━━━━━━━━━ 42s 208ms/step - accuracy: 0.9830 - loss: 0.1334 - val_accuracy: 0.9771 - val_loss: 0.1683
Epoch 9/50
201/201 ━━━━━━━━━━━━━━━━━━━━ 41s 202ms/step - accuracy: 0.9834 - loss: 0.1251 - val_accuracy: 0.9765 - val_loss: 0.1687
Epoch 10/50
201/201 ━━━━━━━━━━━━━━━━━━━━ 41s 202ms/step - accuracy: 0.9844 - loss: 0.1180 - val_accuracy: 0.9763 - val_loss: 0.1746
Epoch 11/50
201/201 ━━━━━━━━━━━━━━━━━━━━ 41s 203ms/step - accuracy: 0.9851 - loss: 0.1117 - val_accuracy: 0.9766 - val_loss: 0.1695
Epoch 12/50
201/201 ━━━━━━━━━━━━━━━━━━━━ 0s 199ms/step - accuracy: 0.9858 - loss: 0.1050

201/201 ━━━━━━━━━━━━━━━━━━━━ 42s 209ms/step - accuracy: 0.9858 - loss: 0.1050 - val_accuracy: 0.9777 - val_loss: 0.1615
Epoch 13/50
201/201 ━━━━━━━━━━━━━━━━━━━━ 41s 203ms/step - accuracy: 0.9866 - loss: 0.0997 - val_accuracy: 0.9772 - val_loss: 0.1668
Epoch 14/50
201/201 ━━━━━━━━━━━━━━━━━━━━ 41s 203ms/step - accuracy: 0.9870 - loss: 0.0959 - val_accuracy: 0.9767 - val_loss: 0.1730
Epoch 15/50
201/201 ━━━━━━━━━━━━━━━━━━━━ 41s 202ms/step - accuracy: 0.9878 - loss: 0.0897 - val_accuracy: 0.9769 - val_loss: 0.1719
Epoch 16/50
201/201 ━━━━━━━━━━━━━━━━━━━━ 41s 203ms/step - accuracy: 0.9878 - loss: 0.0887 - val_accuracy: 0.9773 - val_loss: 0.1666
Epoch 17/50
201/201 ━━━━━━━━━━━━━━━━━━━━ 41s 202ms/step - accuracy: 0.9878 - loss: 0.0877 - val_accuracy: 0.9772 - val_loss: 0.1657
Epoch 18/50
201/201 ━━━━━━━━━━━━━━━━━━━━ 41s 202ms/step - accuracy: 0.9884 - loss: 0.0825 - val_accuracy: 0.9774 - val_loss: 0.1658
Epoch 19/50
201/201 ━━━━━━━━━━━━━━━━━━━━ 41s 201ms/step - accuracy: 0.9891 - loss: 0.079

##Kernel 5 Blur 3

In [ ]:
# ===============================================
# 0. Imports
# ===============================================
import os, zipfile
from glob import glob
import cv2
import numpy as np
import matplotlib.pyplot as plt
from tqdm import tqdm
import tensorflow as tf
from tensorflow.keras import layers, models
from sklearn.model_selection import KFold
import tensorflow.keras.backend as K
from sklearn.utils import shuffle
import albumentations as A

# ✅ Enable mixed precision
from tensorflow.keras import mixed_precision
mixed_precision.set_global_policy("mixed_float16")

# ===============================================
# 1. Unzip Dataset + Auto Path Fix
# ===============================================
zip_path = "/content/stage1_train.zip"
extract_root = "/content/stage1_train"
os.makedirs(extract_root, exist_ok=True)

if not any(os.path.isdir(os.path.join(extract_root, d)) for d in os.listdir(extract_root)):
    print("📦 Extracting dataset...")
    with zipfile.ZipFile(zip_path, 'r') as zip_ref:
        zip_ref.extractall(extract_root)
    print("✅ Extraction completed.")
else:
    print("✅ Dataset already extracted.")

candidates = [d for d in glob(os.path.join(extract_root, "*")) if os.path.isdir(d)]
extract_path = candidates[0] if len(candidates) == 1 else extract_root
print("✅ Using dataset folder:", extract_path)
print("🔍 First 5 contents:", os.listdir(extract_path)[:5])

# ===============================================
# 2. Load Data Function
# ===============================================
def load_data(data_dir, img_size=(512, 512)):
    images, masks = [], []
    all_ids = [d for d in glob(os.path.join(data_dir, "*")) if os.path.isdir(d)]
    print(f"🔍 Found {len(all_ids)} patient folders")

    for img_id in tqdm(all_ids, desc="Loading patients"):
        img_path_list = glob(os.path.join(img_id, "images", "*.png"))
        mask_paths = glob(os.path.join(img_id, "masks", "*.png"))
        if len(img_path_list) == 0 or len(mask_paths) == 0:
            continue

        image = cv2.imread(img_path_list[0], cv2.IMREAD_COLOR)
        if image is None:
            continue
        image = cv2.resize(image, img_size)
        image = image / 255.0

        mask = np.zeros(img_size, dtype=np.uint8)
        for mp in mask_paths:
            m = cv2.imread(mp, cv2.IMREAD_GRAYSCALE)
            if m is None:
                continue
            m = cv2.resize(m, img_size)
            mask = np.maximum(mask, m)
        mask = (mask > 127).astype(np.float32)
        mask = np.expand_dims(mask, axis=-1)

        images.append(image)
        masks.append(mask)

    return np.array(images, dtype=np.float32), np.array(masks, dtype=np.float32)

# ===============================================
# 3. Load Dataset
# ===============================================
IMG_SIZE = (512, 512)
images, masks = load_data(extract_path, img_size=IMG_SIZE)
print("✅ Images:", images.shape)
print("✅ Masks:", masks.shape)

# ===============================================
# 4. Loss Functions
# ===============================================
def dice_loss(y_true, y_pred, smooth=1e-6):
    y_true_f = K.flatten(y_true)
    y_pred_f = K.flatten(y_pred)
    intersection = K.sum(y_true_f * y_pred_f)
    return 1 - (2. * intersection + smooth) / (K.sum(y_true_f) + K.sum(y_pred_f) + smooth)

def bce_dice_loss(y_true, y_pred):
    bce = tf.keras.losses.binary_crossentropy(y_true, y_pred)
    dice = dice_loss(y_true, y_pred)
    return bce + dice

# ===============================================
# 5. Convolution Blur + Hybrid Attention (CBA)
# ===============================================
class ConvBlurAttention(layers.Layer):
    def __init__(self, filters, kernel_size=5, reduction=16, activation=tf.nn.leaky_relu):
        super().__init__()
        # --- Convolution & Normalization ---
        self.conv = layers.Conv2D(filters, kernel_size, padding="same", use_bias=False)
        self.bn = layers.BatchNormalization()
        # --- Blur ---
        self.blur = layers.DepthwiseConv2D(
            kernel_size=3,
            padding="same",
            use_bias=False,
            depthwise_initializer=tf.keras.initializers.Constant(1/9)
        )
        # --- Channel Attention (SE) ---
        self.global_pool = layers.GlobalAveragePooling2D()
        self.dense1 = layers.Dense(filters // reduction, activation="relu")
        self.dense2 = layers.Dense(filters, activation="sigmoid")
        # --- Spatial Attention ---
        self.spatial_attn = layers.Conv2D(1, 7, padding="same", activation="sigmoid")
        # --- Activation ---
        self.activation = activation

    def call(self, x):
        # 1️⃣ Convolution + Blur
        conv_out = self.conv(x)
        conv_out = self.bn(conv_out)
        blur_out = self.blur(conv_out)

        # 2️⃣ Channel Attention
        se = self.global_pool(blur_out)
        se = self.dense1(se)
        se = self.dense2(se)
        se = tf.reshape(se, [-1, 1, 1, se.shape[-1]])
        ch_attn = conv_out * se

        # 3️⃣ Spatial Attention
        sp_attn_map = self.spatial_attn(ch_attn)
        sp_attn = ch_attn * sp_attn_map

        # 4️⃣ Combine with blurred path
        out = sp_attn * 0.7 + blur_out * 0.3
        return self.activation(out)

# ===============================================
# 6. Transformer Encoder Components
# ===============================================
class PatchEmbedding(layers.Layer):
    def __init__(self, embed_dim=192):
        super().__init__()
        self.proj = layers.Conv2D(embed_dim, kernel_size=1, strides=1, padding="valid")
        self.flatten = layers.Reshape((-1, embed_dim))

    def call(self, x):
        x = self.proj(x)
        x = self.flatten(x)
        return x

class PositionalEncoding(layers.Layer):
    def __init__(self, num_patches, embed_dim):
        super().__init__()
        self.pos_embed = self.add_weight(
            name="pos_embed",
            shape=(1, num_patches, embed_dim),
            initializer="random_normal"
        )

    def call(self, x):
        return x + self.pos_embed

class TransformerBlock(layers.Layer):
    def __init__(self, embed_dim=192, num_heads=12, mlp_dim=768, dropout=0.1):
        super().__init__()
        self.norm1 = layers.LayerNormalization(epsilon=1e-6)
        self.attn = layers.MultiHeadAttention(num_heads=num_heads, key_dim=embed_dim, dropout=dropout)
        self.drop1 = layers.Dropout(dropout)
        self.norm2 = layers.LayerNormalization(epsilon=1e-6)
        self.mlp = models.Sequential([
            layers.Dense(mlp_dim, activation=tf.nn.gelu),
            layers.Dropout(dropout),
            layers.Dense(embed_dim),
            layers.Dropout(dropout),
        ])

    def call(self, x):
        attn_output = self.attn(self.norm1(x), self.norm1(x))
        x = x + self.drop1(attn_output)
        mlp_output = self.mlp(self.norm2(x))
        return x + mlp_output

def build_transformer_encoder(x, num_layers=4, embed_dim=192, num_heads=12, mlp_dim=768):
    for _ in range(num_layers):
        x = TransformerBlock(embed_dim, num_heads, mlp_dim)(x)
    return x

# ===============================================
# 7. TransUNet with Hybrid CBA
# ===============================================
def build_transunet(input_shape=(512, 512, 3), num_classes=1):
    inputs = layers.Input(shape=input_shape)
    base_model = tf.keras.applications.ResNet50(include_top=False, weights="imagenet", input_tensor=inputs)
    skip1 = base_model.get_layer("conv1_relu").output
    skip2 = base_model.get_layer("conv2_block3_out").output
    skip3 = base_model.get_layer("conv3_block4_out").output
    cnn_feature = base_model.get_layer("conv4_block6_out").output

    patches = PatchEmbedding(embed_dim=192)(cnn_feature)
    num_patches = patches.shape[1]
    patches = PositionalEncoding(num_patches, 192)(patches)
    transformer_out = build_transformer_encoder(patches)

    h, w = cnn_feature.shape[1], cnn_feature.shape[2]
    x = layers.Reshape((h, w, 192))(transformer_out)

    x = layers.Conv2DTranspose(512, 3, strides=2, padding="same")(x)
    x = layers.Concatenate()([x, skip3])
    x = ConvBlurAttention(512)(x)

    x = layers.Conv2DTranspose(256, 3, strides=2, padding="same")(x)
    x = layers.Concatenate()([x, skip2])
    x = ConvBlurAttention(256)(x)

    x = layers.Conv2DTranspose(128, 3, strides=2, padding="same")(x)
    x = layers.Concatenate()([x, skip1])
    x = ConvBlurAttention(128)(x)

    x = layers.Conv2DTranspose(64, 3, strides=2, padding="same")(x)
    x = ConvBlurAttention(64)(x)

    outputs = layers.Conv2D(num_classes, 1, activation="sigmoid", dtype="float32")(x)
    return models.Model(inputs, outputs, name="TransUNet_HybridCBA")

# ===============================================
# 8. Metrics
# ===============================================
def calculate_metrics(y_true, y_pred, threshold=0.5, smooth=1e-6):
    y_pred_bin = (y_pred > threshold).astype(np.float32)
    pixel_acc = np.mean(y_true == y_pred_bin)
    intersection = np.sum(y_true * y_pred_bin)
    dice = (2 * intersection + smooth) / (np.sum(y_true) + np.sum(y_pred_bin) + smooth)
    union = np.sum(y_true) + np.sum(y_pred_bin) - intersection
    iou = (intersection + smooth) / (union + smooth)
    return pixel_acc, dice, iou

# ===============================================
# 9. K-Fold Training
# ===============================================
images, masks = shuffle(images, masks, random_state=42)
num_folds = 5
kf = KFold(n_splits=num_folds, shuffle=True, random_state=42)
fold_metrics = []

augmentations = [
    A.HorizontalFlip(p=1.0),
    A.VerticalFlip(p=1.0),
    A.GaussianBlur(blur_limit=(3,3), sigma_limit=2, p=1.0),
    A.GaussianBlur(blur_limit=(7,7), sigma_limit=4, p=1.0),  # ✅ Added Gaussian Blur σ=4
    A.Rotate(limit=270, interpolation=cv2.INTER_LINEAR, border_mode=cv2.BORDER_REFLECT_101, p=1.0),
]

for fold, (train_idx, val_idx) in enumerate(kf.split(images)):
    print(f"\n📂 Fold {fold+1}/{num_folds}")
    X_train, X_val = images[train_idx], images[val_idx]
    y_train, y_val = masks[train_idx], masks[val_idx]

    aug_X, aug_y = [], []
    print("🔄 Applying augmentations...")
    for img, mask in tqdm(zip(X_train, y_train), total=len(X_train)):
        aug_X.append(img)
        aug_y.append(mask)
        for aug in augmentations:
            augmented = aug(image=(img*255).astype(np.uint8), mask=(mask*255).astype(np.uint8))
            img_aug = augmented["image"].astype(np.float32)/255.0
            mask_aug = (augmented["mask"]>127).astype(np.float32)
            img_aug = np.reshape(img_aug, (512,512,3))
            mask_aug = np.reshape(mask_aug, (512,512,1))
            aug_X.append(img_aug)
            aug_y.append(mask_aug)

    X_train = np.array(aug_X, dtype=np.float32)
    y_train = np.array(aug_y, dtype=np.float32)
    print(f"✅ Augmented training set: {X_train.shape}, {y_train.shape}")

    model = build_transunet(input_shape=(512,512,3))
    model.compile(optimizer=tf.keras.optimizers.Adam(1e-4), loss=bce_dice_loss, metrics=["accuracy"])

    callbacks = [
        tf.keras.callbacks.ModelCheckpoint(
            f"transunet_hybridCBA_k5b3_fold{fold+1}.weights.h5",
            save_best_only=True,
            monitor="val_loss"
        )
    ]

    model.fit(
        X_train, y_train,
        validation_data=(X_val, y_val),
        epochs=50,
        batch_size=16,
        callbacks=callbacks,
        verbose=1
    )

    preds_val = model.predict(X_val)
    pixel_acc, dice_score, iou_score = calculate_metrics(y_val, preds_val)
    print(f"✅ Fold {fold+1} | Pixel Acc: {pixel_acc:.4f} | Dice: {dice_score:.4f} | IoU: {iou_score:.4f}")
    fold_metrics.append((pixel_acc, dice_score, iou_score))

# ===============================================
# 10. Final Results
# ===============================================
fold_metrics = np.array(fold_metrics)
mean_metrics = np.mean(fold_metrics, axis=0)
std_metrics = np.std(fold_metrics, axis=0)
print("\n📊 Final Cross-Validation Results:")
print(f"Pixel Accuracy: {mean_metrics[0]:.4f} ± {std_metrics[0]:.4f}")
print(f"Dice Score:     {mean_metrics[1]:.4f} ± {std_metrics[1]:.4f}")
print(f"IoU Score:      {mean_metrics[2]:.4f} ± {std_metrics[2]:.4f}")

📦 Extracting dataset...
✅ Extraction completed.
✅ Using dataset folder: /content/stage1_train
🔍 First 5 contents: ['c6de542205b891eed5c40e6d8ae3d03a6ca39b26dc445b4dbc64340d4d64dd2d', 'b2c5d8653c621207e97b699e5c4c05d13df4f02d9db3e594b1f0c22e5b746aae', '12f89395ad5d21491ab9cec137e247652451d283064773507d7dc362243c5b8e', '9774c82396327929fea05e40ae153cabf0107178b2ae3e40a5709b409793887e', '6bd330234b763b77796d4804de8e224881c0fc8dd02650fa708b2edfd8c7461f']
🔍 Found 670 patient folders


Loading patients: 100%|██████████| 670/670 [00:29<00:00, 22.76it/s]


✅ Images: (670, 512, 512, 3)
✅ Masks: (670, 512, 512, 1)

📂 Fold 1/5
🔄 Applying augmentations...


100%|██████████| 536/536 [00:12<00:00, 42.10it/s]


✅ Augmented training set: (3216, 512, 512, 3), (3216, 512, 512, 1)
94765736/94765736 ━━━━━━━━━━━━━━━━━━━━ 5s 0us/step
Epoch 1/50
201/201 ━━━━━━━━━━━━━━━━━━━━ 0s 241ms/step - accuracy: 0.9229 - loss: 0.7315

...

201/201 ━━━━━━━━━━━━━━━━━━━━ 52s 256ms/step - accuracy: 0.9802 - loss: 0.1826 - val_accuracy: 0.9073 - val_loss: 0.8556
Epoch 6/50
201/201 ━━━━━━━━━━━━━━━━━━━━ 0s 244ms/step - accuracy: 0.9797 - loss: 0.1661

201/201 ━━━━━━━━━━━━━━━━━━━━ 52s 256ms/step - accuracy: 0.9797 - loss: 0.1661 - val_accuracy: 0.9734 - val_loss: 0.2189
Epoch 7/50
201/201 ━━━━━━━━━━━━━━━━━━━━ 0s 244ms/step - accuracy: 0.9820 - loss: 0.1473

201/201 ━━━━━━━━━━━━━━━━━━━━ 52s 256ms/step - accuracy: 0.9820 - loss: 0.1473 - val_accuracy: 0.9766 - val_loss: 0.1820
Epoch 8/50
201/201 ━━━━━━━━━━━━━━━━━━━━ 0s 244ms/step - accuracy: 0.9823 - loss: 0.1375

201/201 ━━━━━━━━━━━━━━━━━━━━ 52s 256ms/step - accuracy: 0.9823 - loss: 0.1374 - val_accuracy: 0.9763 - val_loss: 0.1790
Epoch 9/50
201/201 ━━━━━━━━━━━━━━━━━━━━ 50s 249ms/step - accuracy: 0.9838 - loss: 0.1238 - val_accuracy: 0.9765 - val_loss: 0.1817
Epoch 10/50
201/201 ━━━━━━━━━━━━━━━━━━━━ 50s 248ms/step - accuracy: 0.9842 - loss: 0.1196 - val_accuracy: 0.9759 - val_loss: 0.1825
Epoch 11/50
201/201 ━━━━━━━━━━━━━━━━━━━━ 0s 244ms/step - accuracy: 0.9849 - loss: 0.1115

201/201 ━━━━━━━━━━━━━━━━━━━━ 52s 257ms/step - accuracy: 0.9849 - loss: 0.1115 - val_accuracy: 0.9762 - val_loss: 0.1785
Epoch 12/50
201/201 ━━━━━━━━━━━━━━━━━━━━ 50s 249ms/step - accuracy: 0.9856 - loss: 0.1056 - val_accuracy: 0.9739 - val_loss: 0.1926
Epoch 13/50
201/201 ━━━━━━━━━━━━━━━━━━━━ 50s 248ms/step - accuracy: 0.9865 - loss: 0.0998 - val_accuracy: 0.9757 - val_loss: 0.1818
Epoch 14/50
201/201 ━━━━━━━━━━━━━━━━━━━━ 50s 248ms/step - accuracy: 0.9870 - loss: 0.0946 - val_accuracy: 0.9762 - val_loss: 0.1807
Epoch 15/50
201/201 ━━━━━━━━━━━━━━━━━━━━ 50s 248ms/step - accuracy: 0.9880 - loss: 0.0894 - val_accuracy: 0.9762 - val_loss: 0.1836
Epoch 16/50
201/201 ━━━━━━━━━━━━━━━━━━━━ 50s 249ms/step - accuracy: 0.9879 - loss: 0.0892 - val_accuracy: 0.9765 - val_loss: 0.1818
Epoch 17/50
201/201 ━━━━━━━━━━━━━━━━━━━━ 50s 248ms/step - accuracy: 0.9883 - loss: 0.0867 - val_accuracy: 0.9760 - val_loss: 0.1865
Epoch 18/50
201/201 ━━━━━━━━━━━━━━━━━━━━ 50s 247ms/step - accuracy: 0.9886 - loss: 0.082

##Kernel 7 Blur 3

In [ ]:
# ===============================================
# 0. Imports
# ===============================================
import os, zipfile
from glob import glob
import cv2
import numpy as np
import matplotlib.pyplot as plt
from tqdm import tqdm
import tensorflow as tf
from tensorflow.keras import layers, models
from sklearn.model_selection import KFold
import tensorflow.keras.backend as K
from sklearn.utils import shuffle
import albumentations as A

# ✅ Enable mixed precision
from tensorflow.keras import mixed_precision
mixed_precision.set_global_policy("mixed_float16")

# ===============================================
# 1. Unzip Dataset + Auto Path Fix
# ===============================================
zip_path = "/content/stage1_train.zip"
extract_root = "/content/stage1_train"
os.makedirs(extract_root, exist_ok=True)

if not any(os.path.isdir(os.path.join(extract_root, d)) for d in os.listdir(extract_root)):
    print("📦 Extracting dataset...")
    with zipfile.ZipFile(zip_path, 'r') as zip_ref:
        zip_ref.extractall(extract_root)
    print("✅ Extraction completed.")
else:
    print("✅ Dataset already extracted.")

candidates = [d for d in glob(os.path.join(extract_root, "*")) if os.path.isdir(d)]
extract_path = candidates[0] if len(candidates) == 1 else extract_root
print("✅ Using dataset folder:", extract_path)
print("🔍 First 5 contents:", os.listdir(extract_path)[:5])

# ===============================================
# 2. Load Data Function
# ===============================================
def load_data(data_dir, img_size=(512, 512)):
    images, masks = [], []
    all_ids = [d for d in glob(os.path.join(data_dir, "*")) if os.path.isdir(d)]
    print(f"🔍 Found {len(all_ids)} patient folders")

    for img_id in tqdm(all_ids, desc="Loading patients"):
        img_path_list = glob(os.path.join(img_id, "images", "*.png"))
        mask_paths = glob(os.path.join(img_id, "masks", "*.png"))
        if len(img_path_list) == 0 or len(mask_paths) == 0:
            continue

        image = cv2.imread(img_path_list[0], cv2.IMREAD_COLOR)
        if image is None:
            continue
        image = cv2.resize(image, img_size)
        image = image / 255.0

        mask = np.zeros(img_size, dtype=np.uint8)
        for mp in mask_paths:
            m = cv2.imread(mp, cv2.IMREAD_GRAYSCALE)
            if m is None:
                continue
            m = cv2.resize(m, img_size)
            mask = np.maximum(mask, m)
        mask = (mask > 127).astype(np.float32)
        mask = np.expand_dims(mask, axis=-1)

        images.append(image)
        masks.append(mask)

    return np.array(images, dtype=np.float32), np.array(masks, dtype=np.float32)

# ===============================================
# 3. Load Dataset
# ===============================================
IMG_SIZE = (512, 512)
images, masks = load_data(extract_path, img_size=IMG_SIZE)
print("✅ Images:", images.shape)
print("✅ Masks:", masks.shape)

# ===============================================
# 4. Loss Functions
# ===============================================
def dice_loss(y_true, y_pred, smooth=1e-6):
    y_true_f = K.flatten(y_true)
    y_pred_f = K.flatten(y_pred)
    intersection = K.sum(y_true_f * y_pred_f)
    return 1 - (2. * intersection + smooth) / (K.sum(y_true_f) + K.sum(y_pred_f) + smooth)

def bce_dice_loss(y_true, y_pred):
    bce = tf.keras.losses.binary_crossentropy(y_true, y_pred)
    dice = dice_loss(y_true, y_pred)
    return bce + dice

# ===============================================
# 5. Convolution Blur + Hybrid Attention (CBA)
# ===============================================
class ConvBlurAttention(layers.Layer):
    def __init__(self, filters, kernel_size=7, reduction=16, activation=tf.nn.leaky_relu):
        super().__init__()
        # --- Convolution & Normalization ---
        self.conv = layers.Conv2D(filters, kernel_size, padding="same", use_bias=False)
        self.bn = layers.BatchNormalization()
        # --- Blur ---
        self.blur = layers.DepthwiseConv2D(
            kernel_size=3,
            padding="same",
            use_bias=False,
            depthwise_initializer=tf.keras.initializers.Constant(1/9)
        )
        # --- Channel Attention (SE) ---
        self.global_pool = layers.GlobalAveragePooling2D()
        self.dense1 = layers.Dense(filters // reduction, activation="relu")
        self.dense2 = layers.Dense(filters, activation="sigmoid")
        # --- Spatial Attention ---
        self.spatial_attn = layers.Conv2D(1, 7, padding="same", activation="sigmoid")
        # --- Activation ---
        self.activation = activation

    def call(self, x):
        # 1️⃣ Convolution + Blur
        conv_out = self.conv(x)
        conv_out = self.bn(conv_out)
        blur_out = self.blur(conv_out)

        # 2️⃣ Channel Attention
        se = self.global_pool(blur_out)
        se = self.dense1(se)
        se = self.dense2(se)
        se = tf.reshape(se, [-1, 1, 1, se.shape[-1]])
        ch_attn = conv_out * se

        # 3️⃣ Spatial Attention
        sp_attn_map = self.spatial_attn(ch_attn)
        sp_attn = ch_attn * sp_attn_map

        # 4️⃣ Combine with blurred path
        out = sp_attn * 0.7 + blur_out * 0.3
        return self.activation(out)

# ===============================================
# 6. Transformer Encoder Components
# ===============================================
class PatchEmbedding(layers.Layer):
    def __init__(self, embed_dim=192):
        super().__init__()
        self.proj = layers.Conv2D(embed_dim, kernel_size=1, strides=1, padding="valid")
        self.flatten = layers.Reshape((-1, embed_dim))

    def call(self, x):
        x = self.proj(x)
        x = self.flatten(x)
        return x

class PositionalEncoding(layers.Layer):
    def __init__(self, num_patches, embed_dim):
        super().__init__()
        self.pos_embed = self.add_weight(
            name="pos_embed",
            shape=(1, num_patches, embed_dim),
            initializer="random_normal"
        )

    def call(self, x):
        return x + self.pos_embed

class TransformerBlock(layers.Layer):
    def __init__(self, embed_dim=192, num_heads=12, mlp_dim=768, dropout=0.1):
        super().__init__()
        self.norm1 = layers.LayerNormalization(epsilon=1e-6)
        self.attn = layers.MultiHeadAttention(num_heads=num_heads, key_dim=embed_dim, dropout=dropout)
        self.drop1 = layers.Dropout(dropout)
        self.norm2 = layers.LayerNormalization(epsilon=1e-6)
        self.mlp = models.Sequential([
            layers.Dense(mlp_dim, activation=tf.nn.gelu),
            layers.Dropout(dropout),
            layers.Dense(embed_dim),
            layers.Dropout(dropout),
        ])

    def call(self, x):
        attn_output = self.attn(self.norm1(x), self.norm1(x))
        x = x + self.drop1(attn_output)
        mlp_output = self.mlp(self.norm2(x))
        return x + mlp_output

def build_transformer_encoder(x, num_layers=4, embed_dim=192, num_heads=12, mlp_dim=768):
    for _ in range(num_layers):
        x = TransformerBlock(embed_dim, num_heads, mlp_dim)(x)
    return x

# ===============================================
# 7. TransUNet with Hybrid CBA
# ===============================================
def build_transunet(input_shape=(512, 512, 3), num_classes=1):
    inputs = layers.Input(shape=input_shape)
    base_model = tf.keras.applications.ResNet50(include_top=False, weights="imagenet", input_tensor=inputs)
    skip1 = base_model.get_layer("conv1_relu").output
    skip2 = base_model.get_layer("conv2_block3_out").output
    skip3 = base_model.get_layer("conv3_block4_out").output
    cnn_feature = base_model.get_layer("conv4_block6_out").output

    patches = PatchEmbedding(embed_dim=192)(cnn_feature)
    num_patches = patches.shape[1]
    patches = PositionalEncoding(num_patches, 192)(patches)
    transformer_out = build_transformer_encoder(patches)

    h, w = cnn_feature.shape[1], cnn_feature.shape[2]
    x = layers.Reshape((h, w, 192))(transformer_out)

    x = layers.Conv2DTranspose(512, 3, strides=2, padding="same")(x)
    x = layers.Concatenate()([x, skip3])
    x = ConvBlurAttention(512)(x)

    x = layers.Conv2DTranspose(256, 3, strides=2, padding="same")(x)
    x = layers.Concatenate()([x, skip2])
    x = ConvBlurAttention(256)(x)

    x = layers.Conv2DTranspose(128, 3, strides=2, padding="same")(x)
    x = layers.Concatenate()([x, skip1])
    x = ConvBlurAttention(128)(x)

    x = layers.Conv2DTranspose(64, 3, strides=2, padding="same")(x)
    x = ConvBlurAttention(64)(x)

    outputs = layers.Conv2D(num_classes, 1, activation="sigmoid", dtype="float32")(x)
    return models.Model(inputs, outputs, name="TransUNet_HybridCBA")

# ===============================================
# 8. Metrics
# ===============================================
def calculate_metrics(y_true, y_pred, threshold=0.5, smooth=1e-6):
    y_pred_bin = (y_pred > threshold).astype(np.float32)
    pixel_acc = np.mean(y_true == y_pred_bin)
    intersection = np.sum(y_true * y_pred_bin)
    dice = (2 * intersection + smooth) / (np.sum(y_true) + np.sum(y_pred_bin) + smooth)
    union = np.sum(y_true) + np.sum(y_pred_bin) - intersection
    iou = (intersection + smooth) / (union + smooth)
    return pixel_acc, dice, iou

# ===============================================
# 9. K-Fold Training
# ===============================================
images, masks = shuffle(images, masks, random_state=42)
num_folds = 5
kf = KFold(n_splits=num_folds, shuffle=True, random_state=42)
fold_metrics = []

augmentations = [
    A.HorizontalFlip(p=1.0),
    A.VerticalFlip(p=1.0),
    A.GaussianBlur(blur_limit=(3,3), sigma_limit=2, p=1.0),
    A.GaussianBlur(blur_limit=(7,7), sigma_limit=4, p=1.0),  # ✅ Added Gaussian Blur σ=4
    A.Rotate(limit=270, interpolation=cv2.INTER_LINEAR, border_mode=cv2.BORDER_REFLECT_101, p=1.0),
]

for fold, (train_idx, val_idx) in enumerate(kf.split(images)):
    print(f"\n📂 Fold {fold+1}/{num_folds}")
    X_train, X_val = images[train_idx], images[val_idx]
    y_train, y_val = masks[train_idx], masks[val_idx]

    aug_X, aug_y = [], []
    print("🔄 Applying augmentations...")
    for img, mask in tqdm(zip(X_train, y_train), total=len(X_train)):
        aug_X.append(img)
        aug_y.append(mask)
        for aug in augmentations:
            augmented = aug(image=(img*255).astype(np.uint8), mask=(mask*255).astype(np.uint8))
            img_aug = augmented["image"].astype(np.float32)/255.0
            mask_aug = (augmented["mask"]>127).astype(np.float32)
            img_aug = np.reshape(img_aug, (512,512,3))
            mask_aug = np.reshape(mask_aug, (512,512,1))
            aug_X.append(img_aug)
            aug_y.append(mask_aug)

    X_train = np.array(aug_X, dtype=np.float32)
    y_train = np.array(aug_y, dtype=np.float32)
    print(f"✅ Augmented training set: {X_train.shape}, {y_train.shape}")

    model = build_transunet(input_shape=(512,512,3))
    model.compile(optimizer=tf.keras.optimizers.Adam(1e-4), loss=bce_dice_loss, metrics=["accuracy"])

    callbacks = [
        tf.keras.callbacks.ModelCheckpoint(
            f"transunet_hybridCBA_k7b3_fold{fold+1}.weights.h5",
            save_best_only=True,
            monitor="val_loss"
        )
    ]

    model.fit(
        X_train, y_train,
        validation_data=(X_val, y_val),
        epochs=50,
        batch_size=16,
        callbacks=callbacks,
        verbose=1
    )

    preds_val = model.predict(X_val)
    pixel_acc, dice_score, iou_score = calculate_metrics(y_val, preds_val)
    print(f"✅ Fold {fold+1} | Pixel Acc: {pixel_acc:.4f} | Dice: {dice_score:.4f} | IoU: {iou_score:.4f}")
    fold_metrics.append((pixel_acc, dice_score, iou_score))

# ===============================================
# 10. Final Results
# ===============================================
fold_metrics = np.array(fold_metrics)
mean_metrics = np.mean(fold_metrics, axis=0)
std_metrics = np.std(fold_metrics, axis=0)
print("\n📊 Final Cross-Validation Results:")
print(f"Pixel Accuracy: {mean_metrics[0]:.4f} ± {std_metrics[0]:.4f}")
print(f"Dice Score:     {mean_metrics[1]:.4f} ± {std_metrics[1]:.4f}")
print(f"IoU Score:      {mean_metrics[2]:.4f} ± {std_metrics[2]:.4f}")

📦 Extracting dataset...
✅ Extraction completed.
✅ Using dataset folder: /content/stage1_train
🔍 First 5 contents: ['8f6597cd978c060378177df76e554d0578b97eab471e237dbe0adc0dd0d93d63', 'bc115ff727e997a88f7cfe4ce817745731a6c753cb9fab6a36e7e66b415a1d3d', 'b909aa8f6f4bec37c3fb6ff5a85d166162d07983506fcc57be742b0f9dbafbf7', '797945873ca2a95f028671714b71eb3f883efe9dae7fcd3fc0ea1521efb73aaa', 'a486f6ed4b8781e7883e433d06a83dd66db3e8b36d45b9976c4214820ee22629']
🔍 Found 670 patient folders


Loading patients: 100%|██████████| 670/670 [00:28<00:00, 23.24it/s]


✅ Images: (670, 512, 512, 3)
✅ Masks: (670, 512, 512, 1)

📂 Fold 1/5
🔄 Applying augmentations...


100%|██████████| 536/536 [00:12<00:00, 42.86it/s]


✅ Augmented training set: (3216, 512, 512, 3), (3216, 512, 512, 1)
94765736/94765736 ━━━━━━━━━━━━━━━━━━━━ 3s 0us/step
Epoch 1/50
201/201 ━━━━━━━━━━━━━━━━━━━━ 0s 308ms/step - accuracy: 0.9313 - loss: 0.7330

...

201/201 ━━━━━━━━━━━━━━━━━━━━ 66s 326ms/step - accuracy: 0.9796 - loss: 0.1711 - val_accuracy: 0.9350 - val_loss: 0.6478
Epoch 6/50
201/201 ━━━━━━━━━━━━━━━━━━━━ 0s 312ms/step - accuracy: 0.9809 - loss: 0.1484

201/201 ━━━━━━━━━━━━━━━━━━━━ 66s 327ms/step - accuracy: 0.9809 - loss: 0.1484 - val_accuracy: 0.9788 - val_loss: 0.1815
Epoch 7/50
201/201 ━━━━━━━━━━━━━━━━━━━━ 0s 311ms/step - accuracy: 0.9811 - loss: 0.1425

201/201 ━━━━━━━━━━━━━━━━━━━━ 66s 327ms/step - accuracy: 0.9811 - loss: 0.1425 - val_accuracy: 0.9798 - val_loss: 0.1562
Epoch 8/50
201/201 ━━━━━━━━━━━━━━━━━━━━ 64s 316ms/step - accuracy: 0.9821 - loss: 0.1327 - val_accuracy: 0.9797 - val_loss: 0.1563
Epoch 9/50
201/201 ━━━━━━━━━━━━━━━━━━━━ 64s 317ms/step - accuracy: 0.9827 - loss: 0.1271 - val_accuracy: 0.9791 - val_loss: 0.1572
Epoch 10/50
201/201 ━━━━━━━━━━━━━━━━━━━━ 64s 317ms/step - accuracy: 0.9837 - loss: 0.1190 - val_accuracy: 0.9757 - val_loss: 0.1801
Epoch 11/50
201/201 ━━━━━━━━━━━━━━━━━━━━ 0s 313ms/step - accuracy: 0.9844 - loss: 0.1131

201/201 ━━━━━━━━━━━━━━━━━━━━ 66s 328ms/step - accuracy: 0.9844 - loss: 0.1131 - val_accuracy: 0.9802 - val_loss: 0.1497
Epoch 12/50
201/201 ━━━━━━━━━━━━━━━━━━━━ 64s 317ms/step - accuracy: 0.9852 - loss: 0.1052 - val_accuracy: 0.9790 - val_loss: 0.1596
Epoch 13/50
201/201 ━━━━━━━━━━━━━━━━━━━━ 0s 312ms/step - accuracy: 0.9854 - loss: 0.1041

201/201 ━━━━━━━━━━━━━━━━━━━━ 66s 328ms/step - accuracy: 0.9854 - loss: 0.1041 - val_accuracy: 0.9819 - val_loss: 0.1383
Epoch 14/50
201/201 ━━━━━━━━━━━━━━━━━━━━ 64s 317ms/step - accuracy: 0.9865 - loss: 0.0957 - val_accuracy: 0.9802 - val_loss: 0.1517
Epoch 15/50
201/201 ━━━━━━━━━━━━━━━━━━━━ 64s 317ms/step - accuracy: 0.9874 - loss: 0.0902 - val_accuracy: 0.9799 - val_loss: 0.1574
Epoch 16/50
201/201 ━━━━━━━━━━━━━━━━━━━━ 64s 317ms/step - accuracy: 0.9874 - loss: 0.0899 - val_accuracy: 0.9812 - val_loss: 0.1471
Epoch 17/50
201/201 ━━━━━━━━━━━━━━━━━━━━ 64s 317ms/step - accuracy: 0.9882 - loss: 0.0839 - val_accuracy: 0.9782 - val_loss: 0.1755
Epoch 18/50
201/201 ━━━━━━━━━━━━━━━━━━━━ 64s 318ms/step - accuracy: 0.9879 - loss: 0.0854 - val_accuracy: 0.9807 - val_loss: 0.1548
Epoch 19/50
201/201 ━━━━━━━━━━━━━━━━━━━━ 64s 317ms/step - accuracy: 0.9894 - loss: 0.0769 - val_accuracy: 0.9805 - val_loss: 0.1576
Epoch 20/50
201/201 ━━━━━━━━━━━━━━━━━━━━ 64s 317ms/step - accuracy: 0.9890 - loss: 0.077

##Kernel 5, Blur 7

In [ ]:
# ===============================================
# 0. Imports
# ===============================================
import os, zipfile
from glob import glob
import cv2
import numpy as np
import matplotlib.pyplot as plt
from tqdm import tqdm
import tensorflow as tf
from tensorflow.keras import layers, models
from sklearn.model_selection import KFold
import tensorflow.keras.backend as K
from sklearn.utils import shuffle
import albumentations as A

# ✅ Enable mixed precision
from tensorflow.keras import mixed_precision
mixed_precision.set_global_policy("mixed_float16")

# ===============================================
# 1. Unzip Dataset + Auto Path Fix
# ===============================================
zip_path = "/content/stage1_train.zip"
extract_root = "/content/stage1_train"
os.makedirs(extract_root, exist_ok=True)

if not any(os.path.isdir(os.path.join(extract_root, d)) for d in os.listdir(extract_root)):
    print("📦 Extracting dataset...")
    with zipfile.ZipFile(zip_path, 'r') as zip_ref:
        zip_ref.extractall(extract_root)
    print("✅ Extraction completed.")
else:
    print("✅ Dataset already extracted.")

candidates = [d for d in glob(os.path.join(extract_root, "*")) if os.path.isdir(d)]
extract_path = candidates[0] if len(candidates) == 1 else extract_root
print("✅ Using dataset folder:", extract_path)
print("🔍 First 5 contents:", os.listdir(extract_path)[:5])

# ===============================================
# 2. Load Data Function
# ===============================================
def load_data(data_dir, img_size=(512, 512)):
    images, masks = [], []
    all_ids = [d for d in glob(os.path.join(data_dir, "*")) if os.path.isdir(d)]
    print(f"🔍 Found {len(all_ids)} patient folders")

    for img_id in tqdm(all_ids, desc="Loading patients"):
        img_path_list = glob(os.path.join(img_id, "images", "*.png"))
        mask_paths = glob(os.path.join(img_id, "masks", "*.png"))
        if len(img_path_list) == 0 or len(mask_paths) == 0:
            continue

        image = cv2.imread(img_path_list[0], cv2.IMREAD_COLOR)
        if image is None:
            continue
        image = cv2.resize(image, img_size)
        image = image / 255.0

        mask = np.zeros(img_size, dtype=np.uint8)
        for mp in mask_paths:
            m = cv2.imread(mp, cv2.IMREAD_GRAYSCALE)
            if m is None:
                continue
            m = cv2.resize(m, img_size)
            mask = np.maximum(mask, m)
        mask = (mask > 127).astype(np.float32)
        mask = np.expand_dims(mask, axis=-1)

        images.append(image)
        masks.append(mask)

    return np.array(images, dtype=np.float32), np.array(masks, dtype=np.float32)

# ===============================================
# 3. Load Dataset
# ===============================================
IMG_SIZE = (512, 512)
images, masks = load_data(extract_path, img_size=IMG_SIZE)
print("✅ Images:", images.shape)
print("✅ Masks:", masks.shape)

# ===============================================
# 4. Loss Functions
# ===============================================
def dice_loss(y_true, y_pred, smooth=1e-6):
    y_true_f = K.flatten(y_true)
    y_pred_f = K.flatten(y_pred)
    intersection = K.sum(y_true_f * y_pred_f)
    return 1 - (2. * intersection + smooth) / (K.sum(y_true_f) + K.sum(y_pred_f) + smooth)

def bce_dice_loss(y_true, y_pred):
    bce = tf.keras.losses.binary_crossentropy(y_true, y_pred)
    dice = dice_loss(y_true, y_pred)
    return bce + dice

# ===============================================
# 5. Convolution Blur + Hybrid Attention (CBA)
# ===============================================
class ConvBlurAttention(layers.Layer):
    def __init__(self, filters, kernel_size=5, reduction=16, activation=tf.nn.leaky_relu):
        super().__init__()
        # --- Convolution & Normalization ---
        self.conv = layers.Conv2D(filters, kernel_size, padding="same", use_bias=False)
        self.bn = layers.BatchNormalization()
        # --- Blur ---
        self.blur = layers.DepthwiseConv2D(
            kernel_size=7,
            padding="same",
            use_bias=False,
            depthwise_initializer=tf.keras.initializers.Constant(1/9)
        )
        # --- Channel Attention (SE) ---
        self.global_pool = layers.GlobalAveragePooling2D()
        self.dense1 = layers.Dense(filters // reduction, activation="relu")
        self.dense2 = layers.Dense(filters, activation="sigmoid")
        # --- Spatial Attention ---
        self.spatial_attn = layers.Conv2D(1, 7, padding="same", activation="sigmoid")
        # --- Activation ---
        self.activation = activation

    def call(self, x):
        # 1️⃣ Convolution + Blur
        conv_out = self.conv(x)
        conv_out = self.bn(conv_out)
        blur_out = self.blur(conv_out)

        # 2️⃣ Channel Attention
        se = self.global_pool(blur_out)
        se = self.dense1(se)
        se = self.dense2(se)
        se = tf.reshape(se, [-1, 1, 1, se.shape[-1]])
        ch_attn = conv_out * se

        # 3️⃣ Spatial Attention
        sp_attn_map = self.spatial_attn(ch_attn)
        sp_attn = ch_attn * sp_attn_map

        # 4️⃣ Combine with blurred path
        out = sp_attn * 0.7 + blur_out * 0.3
        return self.activation(out)

# ===============================================
# 6. Transformer Encoder Components
# ===============================================
class PatchEmbedding(layers.Layer):
    def __init__(self, embed_dim=192):
        super().__init__()
        self.proj = layers.Conv2D(embed_dim, kernel_size=1, strides=1, padding="valid")
        self.flatten = layers.Reshape((-1, embed_dim))

    def call(self, x):
        x = self.proj(x)
        x = self.flatten(x)
        return x

class PositionalEncoding(layers.Layer):
    def __init__(self, num_patches, embed_dim):
        super().__init__()
        self.pos_embed = self.add_weight(
            name="pos_embed",
            shape=(1, num_patches, embed_dim),
            initializer="random_normal"
        )

    def call(self, x):
        return x + self.pos_embed

class TransformerBlock(layers.Layer):
    def __init__(self, embed_dim=192, num_heads=12, mlp_dim=768, dropout=0.1):
        super().__init__()
        self.norm1 = layers.LayerNormalization(epsilon=1e-6)
        self.attn = layers.MultiHeadAttention(num_heads=num_heads, key_dim=embed_dim, dropout=dropout)
        self.drop1 = layers.Dropout(dropout)
        self.norm2 = layers.LayerNormalization(epsilon=1e-6)
        self.mlp = models.Sequential([
            layers.Dense(mlp_dim, activation=tf.nn.gelu),
            layers.Dropout(dropout),
            layers.Dense(embed_dim),
            layers.Dropout(dropout),
        ])

    def call(self, x):
        attn_output = self.attn(self.norm1(x), self.norm1(x))
        x = x + self.drop1(attn_output)
        mlp_output = self.mlp(self.norm2(x))
        return x + mlp_output

def build_transformer_encoder(x, num_layers=4, embed_dim=192, num_heads=12, mlp_dim=768):
    for _ in range(num_layers):
        x = TransformerBlock(embed_dim, num_heads, mlp_dim)(x)
    return x

# ===============================================
# 7. TransUNet with Hybrid CBA
# ===============================================
def build_transunet(input_shape=(512, 512, 3), num_classes=1):
    inputs = layers.Input(shape=input_shape)
    base_model = tf.keras.applications.ResNet50(include_top=False, weights="imagenet", input_tensor=inputs)
    skip1 = base_model.get_layer("conv1_relu").output
    skip2 = base_model.get_layer("conv2_block3_out").output
    skip3 = base_model.get_layer("conv3_block4_out").output
    cnn_feature = base_model.get_layer("conv4_block6_out").output

    patches = PatchEmbedding(embed_dim=192)(cnn_feature)
    num_patches = patches.shape[1]
    patches = PositionalEncoding(num_patches, 192)(patches)
    transformer_out = build_transformer_encoder(patches)

    h, w = cnn_feature.shape[1], cnn_feature.shape[2]
    x = layers.Reshape((h, w, 192))(transformer_out)

    x = layers.Conv2DTranspose(512, 3, strides=2, padding="same")(x)
    x = layers.Concatenate()([x, skip3])
    x = ConvBlurAttention(512)(x)

    x = layers.Conv2DTranspose(256, 3, strides=2, padding="same")(x)
    x = layers.Concatenate()([x, skip2])
    x = ConvBlurAttention(256)(x)

    x = layers.Conv2DTranspose(128, 3, strides=2, padding="same")(x)
    x = layers.Concatenate()([x, skip1])
    x = ConvBlurAttention(128)(x)

    x = layers.Conv2DTranspose(64, 3, strides=2, padding="same")(x)
    x = ConvBlurAttention(64)(x)

    outputs = layers.Conv2D(num_classes, 1, activation="sigmoid", dtype="float32")(x)
    return models.Model(inputs, outputs, name="TransUNet_HybridCBA")

# ===============================================
# 8. Metrics
# ===============================================
def calculate_metrics(y_true, y_pred, threshold=0.5, smooth=1e-6):
    y_pred_bin = (y_pred > threshold).astype(np.float32)
    pixel_acc = np.mean(y_true == y_pred_bin)
    intersection = np.sum(y_true * y_pred_bin)
    dice = (2 * intersection + smooth) / (np.sum(y_true) + np.sum(y_pred_bin) + smooth)
    union = np.sum(y_true) + np.sum(y_pred_bin) - intersection
    iou = (intersection + smooth) / (union + smooth)
    return pixel_acc, dice, iou

# ===============================================
# 9. K-Fold Training
# ===============================================
images, masks = shuffle(images, masks, random_state=42)
num_folds = 5
kf = KFold(n_splits=num_folds, shuffle=True, random_state=42)
fold_metrics = []

augmentations = [
    A.HorizontalFlip(p=1.0),
    A.VerticalFlip(p=1.0),
    A.GaussianBlur(blur_limit=(3,3), sigma_limit=2, p=1.0),
    A.GaussianBlur(blur_limit=(7,7), sigma_limit=4, p=1.0),  # ✅ Added Gaussian Blur σ=4
    A.Rotate(limit=270, interpolation=cv2.INTER_LINEAR, border_mode=cv2.BORDER_REFLECT_101, p=1.0),
]

for fold, (train_idx, val_idx) in enumerate(kf.split(images)):
    print(f"\n📂 Fold {fold+1}/{num_folds}")
    X_train, X_val = images[train_idx], images[val_idx]
    y_train, y_val = masks[train_idx], masks[val_idx]

    aug_X, aug_y = [], []
    print("🔄 Applying augmentations...")
    for img, mask in tqdm(zip(X_train, y_train), total=len(X_train)):
        aug_X.append(img)
        aug_y.append(mask)
        for aug in augmentations:
            augmented = aug(image=(img*255).astype(np.uint8), mask=(mask*255).astype(np.uint8))
            img_aug = augmented["image"].astype(np.float32)/255.0
            mask_aug = (augmented["mask"]>127).astype(np.float32)
            img_aug = np.reshape(img_aug, (512,512,3))
            mask_aug = np.reshape(mask_aug, (512,512,1))
            aug_X.append(img_aug)
            aug_y.append(mask_aug)

    X_train = np.array(aug_X, dtype=np.float32)
    y_train = np.array(aug_y, dtype=np.float32)
    print(f"✅ Augmented training set: {X_train.shape}, {y_train.shape}")

    model = build_transunet(input_shape=(512,512,3))
    model.compile(optimizer=tf.keras.optimizers.Adam(1e-4), loss=bce_dice_loss, metrics=["accuracy"])

    callbacks = [
        tf.keras.callbacks.ModelCheckpoint(
            f"transunet_hybridCBA_k5b7fold{fold+1}.weights.h5",
            save_best_only=True,
            monitor="val_loss"
        )
    ]

    model.fit(
        X_train, y_train,
        validation_data=(X_val, y_val),
        epochs=50,
        batch_size=16,
        callbacks=callbacks,
        verbose=1
    )

    preds_val = model.predict(X_val)
    pixel_acc, dice_score, iou_score = calculate_metrics(y_val, preds_val)
    print(f"✅ Fold {fold+1} | Pixel Acc: {pixel_acc:.4f} | Dice: {dice_score:.4f} | IoU: {iou_score:.4f}")
    fold_metrics.append((pixel_acc, dice_score, iou_score))

# ===============================================
# 10. Final Results
# ===============================================
fold_metrics = np.array(fold_metrics)
mean_metrics = np.mean(fold_metrics, axis=0)
std_metrics = np.std(fold_metrics, axis=0)
print("\n📊 Final Cross-Validation Results:")
print(f"Pixel Accuracy: {mean_metrics[0]:.4f} ± {std_metrics[0]:.4f}")
print(f"Dice Score:     {mean_metrics[1]:.4f} ± {std_metrics[1]:.4f}")
print(f"IoU Score:      {mean_metrics[2]:.4f} ± {std_metrics[2]:.4f}")

📦 Extracting dataset...
✅ Extraction completed.
✅ Using dataset folder: /content/stage1_train
🔍 First 5 contents: ['c6de542205b891eed5c40e6d8ae3d03a6ca39b26dc445b4dbc64340d4d64dd2d', 'b2c5d8653c621207e97b699e5c4c05d13df4f02d9db3e594b1f0c22e5b746aae', '12f89395ad5d21491ab9cec137e247652451d283064773507d7dc362243c5b8e', '9774c82396327929fea05e40ae153cabf0107178b2ae3e40a5709b409793887e', '6bd330234b763b77796d4804de8e224881c0fc8dd02650fa708b2edfd8c7461f']
🔍 Found 670 patient folders


Loading patients: 100%|██████████| 670/670 [00:29<00:00, 22.79it/s]


✅ Images: (670, 512, 512, 3)
✅ Masks: (670, 512, 512, 1)

📂 Fold 1/5
🔄 Applying augmentations...


100%|██████████| 536/536 [00:12<00:00, 43.66it/s]


✅ Augmented training set: (3216, 512, 512, 3), (3216, 512, 512, 1)
94765736/94765736 ━━━━━━━━━━━━━━━━━━━━ 5s 0us/step
Epoch 1/50
201/201 ━━━━━━━━━━━━━━━━━━━━ 0s 260ms/step - accuracy: 0.9414 - loss: 0.4422

...

201/201 ━━━━━━━━━━━━━━━━━━━━ 55s 274ms/step - accuracy: 0.9805 - loss: 0.1443 - val_accuracy: 0.9080 - val_loss: 0.8379
Epoch 6/50
201/201 ━━━━━━━━━━━━━━━━━━━━ 0s 262ms/step - accuracy: 0.9815 - loss: 0.1346

201/201 ━━━━━━━━━━━━━━━━━━━━ 55s 273ms/step - accuracy: 0.9815 - loss: 0.1346 - val_accuracy: 0.9698 - val_loss: 0.2233
Epoch 7/50
201/201 ━━━━━━━━━━━━━━━━━━━━ 0s 262ms/step - accuracy: 0.9824 - loss: 0.1254

201/201 ━━━━━━━━━━━━━━━━━━━━ 55s 274ms/step - accuracy: 0.9824 - loss: 0.1254 - val_accuracy: 0.9743 - val_loss: 0.1945
Epoch 8/50
201/201 ━━━━━━━━━━━━━━━━━━━━ 0s 262ms/step - accuracy: 0.9823 - loss: 0.1267

201/201 ━━━━━━━━━━━━━━━━━━━━ 55s 273ms/step - accuracy: 0.9823 - loss: 0.1266 - val_accuracy: 0.9764 - val_loss: 0.1763
Epoch 9/50
201/201 ━━━━━━━━━━━━━━━━━━━━ 0s 262ms/step - accuracy: 0.9840 - loss: 0.1164

201/201 ━━━━━━━━━━━━━━━━━━━━ 55s 274ms/step - accuracy: 0.9840 - loss: 0.1164 - val_accuracy: 0.9756 - val_loss: 0.1754
Epoch 10/50
201/201 ━━━━━━━━━━━━━━━━━━━━ 54s 266ms/step - accuracy: 0.9843 - loss: 0.1099 - val_accuracy: 0.9735 - val_loss: 0.2066
Epoch 11/50
201/201 ━━━━━━━━━━━━━━━━━━━━ 54s 266ms/step - accuracy: 0.9851 - loss: 0.1087 - val_accuracy: 0.9758 - val_loss: 0.1863
Epoch 12/50
201/201 ━━━━━━━━━━━━━━━━━━━━ 54s 267ms/step - accuracy: 0.9859 - loss: 0.1014 - val_accuracy: 0.9755 - val_loss: 0.1810
Epoch 13/50
201/201 ━━━━━━━━━━━━━━━━━━━━ 53s 265ms/step - accuracy: 0.9864 - loss: 0.0965 - val_accuracy: 0.9757 - val_loss: 0.1884
Epoch 14/50
201/201 ━━━━━━━━━━━━━━━━━━━━ 54s 266ms/step - accuracy: 0.9865 - loss: 0.0955 - val_accuracy: 0.9761 - val_loss: 0.1845
Epoch 15/50
201/201 ━━━━━━━━━━━━━━━━━━━━ 54s 266ms/step - accuracy: 0.9872 - loss: 0.0902 - val_accuracy: 0.9754 - val_loss: 0.1961
Epoch 16/50
201/201 ━━━━━━━━━━━━━━━━━━━━ 53s 266ms/step - accuracy: 0.9876 - loss: 0.087